# Groundwater levels: cleaning and QC

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Cleans the 13 Jaipur-circle district Excel files (4,135 wells, 128,185 seasonal records: 58,427 numeric, 14,307 dry, 55,451 no reading). Also checks coordinates and recovers the Alwar source records (Steps 2–6).

| | |
|---|---|
| **Reads** | `gw_framework/data/Rajasthan_gw_data/*.xlsx`, `gw_framework/data/gwl/` |
| **Writes** | `Rajasthan_gw_data/Groundwater_Cleaning_According_to_User_Rules/`, `Groundwater_Reconciliation_Audit/`, `gwl/diagnostics/Alwar_Source_Recovery/Step_2…Step_6` |
| **Cells** | 32 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Load the district Excel files
- 2. Cleaning rules, elevations and heads
- 3. Coordinate checks
- 4. Parsed observations in `gwl/` (earlier WRIS/CGWB route)
- 5. Alwar source recovery — Steps 2 to 6

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["geopandas", "geopy", "openpyxl", "pyarrow"], modflow=False)
del TOKEN, URL


## 1. Load the district Excel files

**[orig #9]** Status check of the Rajasthan groundwater folder  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import pandas as pd

GW_DIR = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

print("=" * 70)
print("GROUNDWATER DATA CLEANING STATUS CHECK")
print("=" * 70)

if not GW_DIR.exists():
    print(f"Folder not found: {GW_DIR}")
else:
    files = sorted(
        p for p in GW_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in [".xlsx", ".xls", ".csv"]
    )

    print(f"\nFolder: {GW_DIR}")
    print(f"Data files found: {len(files)}\n")

    for path in files:
        print(f"\nFile: {path.name}")

        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path, nrows=5)
                print("Type: CSV")
                print(f"Columns: {list(df.columns)}")
                print("Preview:")
                print(df.head(3).to_string(index=False))

            else:
                xls = pd.ExcelFile(path)
                print(f"Sheets: {xls.sheet_names}")

                for sheet in xls.sheet_names[:2]:
                    sample = pd.read_excel(
                        path, sheet_name=sheet, nrows=5
                    )
                    print(f"  Sheet: {sheet}")
                    print(f"  Columns: {list(sample.columns)}")
                    print(sample.head(2).to_string(index=False))

        except Exception as e:
            print(f"Could not inspect file: {e}")

print("\n" + "=" * 70)
print("STATUS CHECK COMPLETE — NO FILES WERE MODIFIED")
print("=" * 70)

**[orig #51]** List files in `Rajasthan_gw_data`  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

GW_FOLDER = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

print("Files in Rajasthan groundwater folder:\n")

for p in GW_FOLDER.rglob("*"):
    if p.is_file():
        print(f"{p.name}  |  {p}")

**[orig #52]** Open every district Excel file and list worksheets  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import pandas as pd

GW_FOLDER = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

files = sorted(GW_FOLDER.glob("*.xlsx"))

print("Number of Excel files:", len(files))

for file in files:
    print("\n" + "=" * 70)
    print("FILE:", file.name)

    try:
        xls = pd.ExcelFile(file)
        print("Worksheets:", xls.sheet_names)

        for sheet in xls.sheet_names:
            df = pd.read_excel(file, sheet_name=sheet)

            print(f"\nSheet: {sheet}")
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(3))

    except Exception as e:
        print("Error reading file:", e)

**[orig #54]** Load and combine all district Excel files into `gw`  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

# Folder containing your Rajasthan Jaipur Circle groundwater files
GW_FOLDER = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

# Check that the folder exists
if not GW_FOLDER.exists():
    raise FileNotFoundError(
        f"Folder not found: {GW_FOLDER}\n"
        "Check that Google Drive is mounted."
    )

# Read and combine all Excel files
files = sorted(GW_FOLDER.glob("*.xlsx"))

if not files:
    raise FileNotFoundError("No Excel files found in the folder.")

frames = []

for file in files:
    df = pd.read_excel(file)
    df.columns = [str(c).strip() for c in df.columns]
    df["Source_File"] = file.name

    if "District" in df.columns:
        df["District"] = df["District"].fillna(file.stem)
    else:
        df["District"] = file.stem

    frames.append(df)

gw = pd.concat(frames, ignore_index=True)

# Convert coordinates to numeric
for col in ["Latitude", "Longitude"]:
    if col not in gw.columns:
        raise KeyError(f"Expected column not found: {col}")
    gw[col] = pd.to_numeric(gw[col], errors="coerce")

# Display results
print("Excel files loaded:", len(files))
print("Combined rows:", len(gw))
print("Available columns:", gw.columns.tolist())

display(
    gw[[
        "District",
        "WIMS_ID",
        "Latitude",
        "Longitude"
    ]].head(10)
)

print("\nMissing latitude:", gw["Latitude"].isna().sum())
print("Missing longitude:", gw["Longitude"].isna().sum())

valid = (
    gw["Latitude"].between(-90, 90)
    & gw["Longitude"].between(-180, 180)
)

print("Rows with valid coordinate ranges:", int(valid.sum()), "of", len(gw))


**[orig #58]** Reload district files, fill district from filename  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import re

# 1. Paths
folder = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)
output_folder = folder / "QC_outputs"
output_folder.mkdir(exist_ok=True)

files = sorted(folder.glob("*.xlsx"))
if not files:
    raise FileNotFoundError(f"No Excel files found in {folder}")

# 2. Load all district files
frames = []
for file in files:
    df = pd.read_excel(file)
    df.columns = [str(c).strip() for c in df.columns]
    df["Source_File"] = file.name

    # Fill district from filename only when missing
    if "District" not in df.columns:
        df["District"] = file.stem
    else:
        df["District"] = df["District"].fillna(file.stem)

    frames.append(df)

gw = pd.concat(frames, ignore_index=True, sort=False)

print("Files loaded:", len(files))
print("Total records:", len(gw))
print("Columns:", len(gw.columns))

# 3. Identify groundwater level columns
pattern = re.compile(
    r"^WL[_ ]*(pre|post)[_ ]*(\d{2,4})$",
    flags=re.IGNORECASE
)

wl_columns = {}
for col in gw.columns:
    match = pattern.match(str(col).strip())
    if match:
        season = match.group(1).lower()
        year_text = match.group(2)
        year = int(year_text)
        if year < 100:
            year = 2000 + year if year <= 30 else 1900 + year
        wl_columns[col] = (season, year)

if not wl_columns:
    raise ValueError(
        "No WL_pre/WL_post columns detected. Check actual column names."
    )

print("Groundwater measurement columns found:", len(wl_columns))
print("Years found:", sorted({y for s, y in wl_columns.values()}))

# 4. Basic coordinate and identifier audit
for col in ["Latitude", "Longitude"]:
    if col in gw.columns:
        gw[col] = pd.to_numeric(gw[col], errors="coerce")

id_col = next(
    (c for c in ["WIMS_ID", "ROLTA_ID/UID", "UID", "SSN"]
     if c in gw.columns),
    None
)

audit = {
    "Excel_files_loaded": len(files),
    "Total_rows": len(gw),
    "Unique_well_ids": gw[id_col].nunique() if id_col else None,
    "Missing_well_ids": int(gw[id_col].isna().sum()) if id_col else None,
    "Duplicate_well_id_rows": (
        int(gw[id_col].duplicated(keep=False).sum()) if id_col else None
    ),
    "Missing_latitude": int(gw["Latitude"].isna().sum())
        if "Latitude" in gw.columns else None,
    "Missing_longitude": int(gw["Longitude"].isna().sum())
        if "Longitude" in gw.columns else None,
    "WL_columns_detected": len(wl_columns),
}

# 5. Create a long-format seasonal dataset
id_vars = [c for c in gw.columns if c not in wl_columns]
long_parts = []

for col, (season, year) in wl_columns.items():
    part = gw[id_vars].copy()
    part["Year"] = year
    part["Season"] = season
    part["Groundwater_Level_bgl"] = pd.to_numeric(
        gw[col], errors="coerce"
    )
    part["Original_WL_Column"] = col
    long_parts.append(part)

gw_long = pd.concat(long_parts, ignore_index=True)

# Preserve missing measurements for auditing
missing_by_year_season = (
    gw_long.groupby(["Year", "Season"])["Groundwater_Level_bgl"]
    .agg(Total_records="size", Missing_values=lambda x: x.isna().sum())
    .reset_index()
)

# 6. Duplicate well IDs report
if id_col:
    duplicates = gw[
        gw[id_col].notna() & gw[id_col].duplicated(keep=False)
    ].sort_values(id_col)
else:
    duplicates = pd.DataFrame()

# 7. Save outputs
gw.to_csv(output_folder / "groundwater_combined_wide.csv", index=False)
gw_long.to_csv(output_folder / "groundwater_long_seasonal.csv", index=False)
pd.DataFrame([audit]).to_csv(
    output_folder / "groundwater_QC_summary.csv", index=False
)
missing_by_year_season.to_csv(
    output_folder / "missing_WL_by_year_season.csv", index=False
)
duplicates.to_csv(
    output_folder / "duplicate_well_IDs.csv", index=False
)

print("\nQC SUMMARY")
display(pd.DataFrame([audit]))
print("\nMissing measurements by year and season")
display(missing_by_year_season)
print("\nDuplicate well-ID rows:", len(duplicates))
print("\nSaved outputs in:", output_folder)


**[orig #59]** Value review of water levels by season → `groundwater_value_review.csv`  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import re

# --------------------------------------------------
# 1. Load the original district files
# --------------------------------------------------
folder = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)
out = folder / "QC_outputs"
out.mkdir(parents=True, exist_ok=True)

files = sorted(folder.glob("*.xlsx"))
if not files:
    raise FileNotFoundError(f"No Excel files found: {folder}")

frames = []
for f in files:
    d = pd.read_excel(f)
    d.columns = [str(c).strip() for c in d.columns]
    d["Source_File"] = f.name

    if "District" not in d.columns:
        d["District"] = f.stem
    else:
        d["District"] = d["District"].fillna(f.stem)

    frames.append(d)

gw = pd.concat(frames, ignore_index=True, sort=False)

print(f"Loaded {len(files)} files and {len(gw)} well records.")

# --------------------------------------------------
# 2. Detect groundwater columns
# --------------------------------------------------
pattern = re.compile(r"^WL[_ ]*(pre|post)[_ ]*(\d{2,4})$", re.I)
wl_cols = {}

for col in gw.columns:
    m = pattern.match(col)
    if m:
        season = m.group(1).lower()
        year = int(m.group(2))
        if year < 100:
            year = 2000 + year if year <= 30 else 1900 + year
        wl_cols[col] = (season, year)

if not wl_cols:
    raise ValueError("No WL_pre/WL_post columns were found.")

# Identify well ID and total well depth fields
id_col = next(
    (c for c in ["WIMS_ID", "ROLTA_ID/UID", "UID", "SSN"]
     if c in gw.columns),
    None
)
depth_col = next(
    (c for c in ["Total_Depth_bgl", "Total_Depth_BGL",
                 "Total Depth bgl", "Total_Depth"]
     if c in gw.columns),
    None
)

if id_col is None:
    raise ValueError("Could not identify a well ID column.")

# --------------------------------------------------
# 3. Convert measurements to long format
#    Keep raw values for audit
# --------------------------------------------------
id_vars = [c for c in gw.columns if c not in wl_cols]
parts = []

for col, (season, year) in wl_cols.items():
    part = gw[id_vars].copy()
    part["Year"] = year
    part["Season"] = season
    part["Original_WL_Column"] = col
    part["WL_raw"] = gw[col].values
    part["WL_bgl"] = pd.to_numeric(
        gw[col].replace(r"^\s*$", np.nan, regex=True),
        errors="coerce"
    ).values
    parts.append(part)

long = pd.concat(parts, ignore_index=True)

# Exclude fully empty future column(s) from value checks,
# but keep them in the source data.
observed = long[long["WL_raw"].notna()].copy()
observed = observed[
    observed["WL_raw"].astype(str).str.strip().ne("")
].copy()

# --------------------------------------------------
# 4. Parse total depth, if available
# --------------------------------------------------
if depth_col:
    observed["Total_Depth_numeric"] = pd.to_numeric(
        observed[depth_col], errors="coerce"
    )
else:
    observed["Total_Depth_numeric"] = np.nan

# --------------------------------------------------
# 5. Create review flags
#    Flags are for investigation, not automatic deletion.
# --------------------------------------------------
observed["Flag_non_numeric"] = observed["WL_bgl"].isna()
observed["Flag_negative"] = observed["WL_bgl"].lt(0)
observed["Flag_zero"] = observed["WL_bgl"].eq(0)

# Broad screening threshold only; confirm units and local context.
observed["Flag_over_100m"] = observed["WL_bgl"].gt(100)

# Only compare where both measurements are numeric.
observed["Flag_deeper_than_total_depth"] = (
    observed["WL_bgl"].notna()
    & observed["Total_Depth_numeric"].notna()
    & (observed["WL_bgl"] > observed["Total_Depth_numeric"])
)

flag_cols = [
    "Flag_non_numeric",
    "Flag_negative",
    "Flag_zero",
    "Flag_over_100m",
    "Flag_deeper_than_total_depth"
]
observed["Any_review_flag"] = observed[flag_cols].any(axis=1)

def reasons(row):
    labels = []
    if row["Flag_non_numeric"]:
        labels.append("Non-numeric measurement")
    if row["Flag_negative"]:
        labels.append("Negative groundwater depth")
    if row["Flag_zero"]:
        labels.append("Zero depth - verify")
    if row["Flag_over_100m"]:
        labels.append("Over 100 m - verify units/value")
    if row["Flag_deeper_than_total_depth"]:
        labels.append("WL deeper than recorded total well depth")
    return "; ".join(labels)

observed["Review_reason"] = observed.apply(reasons, axis=1)

review = observed[observed["Any_review_flag"]].copy()

# --------------------------------------------------
# 6. Summary statistics by year and season
# --------------------------------------------------
summary = (
    long.groupby(["Year", "Season"])["WL_bgl"]
    .agg(
        Total_records="size",
        Observed="count",
        Minimum="min",
        Q1=lambda x: x.quantile(0.25),
        Median="median",
        Q3="quantile",
        Maximum="max"
    )
    .reset_index()
)

summary["Missing"] = summary["Total_records"] - summary["Observed"]
summary["Observed_percent"] = (
    100 * summary["Observed"] / summary["Total_records"]
)

# A count of each review type
flag_summary = pd.DataFrame({
    "Review_check": [
        "Non-numeric, nonblank measurement",
        "Negative depth",
        "Zero depth",
        "Depth greater than 100 m",
        "WL deeper than total well depth",
        "Rows with any review flag"
    ],
    "Count": [
        int(observed["Flag_non_numeric"].sum()),
        int(observed["Flag_negative"].sum()),
        int(observed["Flag_zero"].sum()),
        int(observed["Flag_over_100m"].sum()),
        int(observed["Flag_deeper_than_total_depth"].sum()),
        int(observed["Any_review_flag"].sum())
    ]
})

# --------------------------------------------------
# 7. Save reports; no original values are changed
# --------------------------------------------------
review.to_csv(out / "groundwater_value_review.csv", index=False)
summary.to_csv(out / "groundwater_value_summary_by_season.csv", index=False)
flag_summary.to_csv(out / "groundwater_value_flag_summary.csv", index=False)

print("\nVALUE CHECK SUMMARY")
display(flag_summary)

print("\nObserved value ranges by year and season")
display(summary)

print("\nFirst 30 flagged records")
display(
    review[[
        id_col, "District", "Year", "Season",
        "Original_WL_Column", "WL_raw", "WL_bgl",
        depth_col if depth_col else "Source_File",
        "Review_reason"
    ]].head(30)
)

print("\nReview file:", out / "groundwater_value_review.csv")
print("Summary file:", out / "groundwater_value_summary_by_season.csv")
print("Flag counts:", out / "groundwater_value_flag_summary.csv")


## 2. Cleaning rules, elevations and heads

Note: `groundwater_cleaned_long_seasonal.csv` has the level in `WL_bgl` and **no coordinates**. Join coordinates from the well inventory before using it for calibration.

**[orig #10]** **MAIN** — clean water-level readings by your rules (numeric / dry / no reading, flags) → `Groundwater_Cleaning_According_to_User_Rules/`  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# GROUNDWATER MEASUREMENT CLEANING — RAJASTHAN PhD PROJECT
# Rules:
#   1. Preserve original Excel files and raw measurements.
#   2. "Dry" is a separate observation status.
#   3. Values such as 13.10* are converted to 13.10 for analysis.
#   4. Blank, "-", "filled", and unconfirmed text -> No reading.
#   5. Retain numeric values >100 m, even if deeper than well depth.
#   6. Retain zero and negative values, but flag them for review.
#   7. Do not silently correct coordinates or delete observations.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import glob
import numpy as np
import pandas as pd
from collections import Counter

# -------------------- PATHS --------------------

INPUT_DIR = (
    "/content/drive/MyDrive/gw_framework/data/"
    "Rajasthan_gw_data"
)

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Groundwater_Cleaning_According_to_User_Rules"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------- SETTINGS --------------------

# Extend this list if your source files use additional total-depth names.
TOTAL_DEPTH_CANDIDATES = [
    "Total_Depth_bgl",
    "Total_Depth",
    "Total depth",
    "Total_Depth_BGL",
    "Well_Depth",
]

# -------------------- HELPER FUNCTIONS --------------------

def normalize_column_name(value):
    """Normalize column names for matching without changing source headers."""
    return re.sub(r"[^a-z0-9]", "", str(value).strip().lower())


def find_column(df, candidates):
    """Find a column using normalized candidate names."""
    lookup = {
        normalize_column_name(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in lookup:
            return lookup[key]

    return None


def parse_numeric(value):
    """
    Return a numeric measurement where possible.
    For values such as 13.10*, use 13.10.
    Return NaN for nonnumeric values.
    """
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()

    if not text:
        return np.nan

    # Remove commas used as thousands separators.
    text = text.replace(",", "")

    # Accept a numeric value followed by an asterisk, e.g. 13.10*
    # Also accept ordinary signed and decimal numeric strings.
    match = re.fullmatch(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*\*?",
        text
    )

    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return np.nan

    return np.nan


def classify_measurement(value):
    """
    Classify the original observation without deleting it.
    Returns:
      WL_status, WL_bgl, WL_raw_has_asterisk
    """
    if pd.isna(value):
        return "No reading", np.nan, False

    if isinstance(value, str):
        text = value.strip()

        if text == "":
            return "No reading", np.nan, False

        if text.lower() == "dry":
            return "Dry", np.nan, False

        number = parse_numeric(text)

        if pd.notna(number):
            return (
                "Numeric reading",
                number,
                "*" in text
            )

        return "No reading", np.nan, False

    number = parse_numeric(value)

    if pd.notna(number):
        return "Numeric reading", number, False

    return "No reading", np.nan, False


def identify_water_level_columns(columns):
    """
    Identify columns such as:
      WL_pre_11, WL_post_11, WL_Pre_25, WL_Post_25
    """
    results = []

    for col in columns:
        text = str(col).strip()

        match = re.fullmatch(
            r"WL[_\s-]*(pre|post)[_\s-]*(\d{2}|\d{4})",
            text,
            flags=re.IGNORECASE
        )

        if match:
            season = match.group(1).capitalize()
            year_text = match.group(2)

            if len(year_text) == 2:
                year = 2000 + int(year_text)
            else:
                year = int(year_text)

            results.append({
                "column": col,
                "season": season,
                "year": year
            })

    return results


# -------------------- DISCOVER SOURCE EXCEL FILES --------------------

excel_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.xlsx")) +
    glob.glob(os.path.join(INPUT_DIR, "*.xls"))
)

# Ignore temporary Excel lock files.
excel_files = [
    path for path in excel_files
    if not os.path.basename(path).startswith("~$")
]

if not excel_files:
    raise FileNotFoundError(
        f"No .xlsx or .xls source files found in: {INPUT_DIR}"
    )

print(f"Excel files discovered: {len(excel_files)}")
print(f"Output directory: {OUTPUT_DIR}")

# -------------------- PROCESS ALL WORKBOOKS --------------------

all_wide = []
all_long = []
file_reports = []
column_reports = []
review_records = []

for file_path in excel_files:

    filename = os.path.basename(file_path)
    district_or_source = os.path.splitext(filename)[0]

    print(f"Processing: {filename}")

    try:
        workbook = pd.ExcelFile(file_path)
    except Exception as exc:
        file_reports.append({
            "Source_file": filename,
            "Sheet": None,
            "Rows": 0,
            "Water_level_columns": 0,
            "Status": "FILE_READ_ERROR",
            "Details": str(exc)
        })
        continue

    for sheet_name in workbook.sheet_names:

        try:
            df = pd.read_excel(
                file_path,
                sheet_name=sheet_name,
                dtype=object
            )
        except Exception as exc:
            file_reports.append({
                "Source_file": filename,
                "Sheet": sheet_name,
                "Rows": 0,
                "Water_level_columns": 0,
                "Status": "SHEET_READ_ERROR",
                "Details": str(exc)
            })
            continue

        if df.empty:
            file_reports.append({
                "Source_file": filename,
                "Sheet": sheet_name,
                "Rows": 0,
                "Water_level_columns": 0,
                "Status": "EMPTY_SHEET",
                "Details": ""
            })
            continue

        # Keep source columns as read; create a separate derived dataset.
        df = df.copy()
        df["Source_file"] = filename
        df["Source_sheet"] = sheet_name
        df["Source_row_number"] = np.arange(2, len(df) + 2)
        df["Source_district_or_file"] = district_or_source

        wl_columns = identify_water_level_columns(df.columns)

        if not wl_columns:
            file_reports.append({
                "Source_file": filename,
                "Sheet": sheet_name,
                "Rows": len(df),
                "Water_level_columns": 0,
                "Status": "NO_WL_COLUMNS_RECOGNIZED",
                "Details": (
                    "Check whether water-level columns follow "
                    "the WL_pre_YY / WL_post_YY naming pattern."
                )
            })
            continue

        # Locate total well depth, if present.
        total_depth_col = find_column(
            df,
            TOTAL_DEPTH_CANDIDATES
        )

        if total_depth_col:
            total_depth_numeric = df[total_depth_col].apply(parse_numeric)
        else:
            total_depth_numeric = pd.Series(
                np.nan,
                index=df.index,
                dtype=float
            )

        # Build a stable row identifier for this source record.
        # Original well-ID columns are preserved.
        well_id_col = find_column(
            df,
            [
                "Well_ID", "Well ID", "WellID",
                "Unique_ID", "UniqueID", "ID",
                "Well_No", "Well Number",
                "Monitoring_Well_ID"
            ]
        )

        if well_id_col:
            source_well_id = df[well_id_col].copy()
        else:
            source_well_id = pd.Series(
                [
                    f"{filename}::{sheet_name}::row{row_num}"
                    for row_num in df["Source_row_number"]
                ],
                index=df.index
            )

        # Long table: one row per source well and seasonal measurement.
        for wl_info in wl_columns:

            wl_col = wl_info["column"]

            column_reports.append({
                "Source_file": filename,
                "Sheet": sheet_name,
                "Original_WL_Column": str(wl_col),
                "Year": wl_info["year"],
                "Season": wl_info["season"]
            })

            for idx in df.index:

                raw_value = df.at[idx, wl_col]

                status, numeric_value, has_asterisk = (
                    classify_measurement(raw_value)
                )

                total_depth = total_depth_numeric.loc[idx]

                is_numeric = pd.notna(numeric_value)
                is_negative = bool(is_numeric and numeric_value < 0)
                is_zero = bool(is_numeric and numeric_value == 0)
                is_over_100 = bool(
                    is_numeric and numeric_value > 100
                )
                is_deeper_than_well = bool(
                    is_numeric
                    and pd.notna(total_depth)
                    and numeric_value > total_depth
                )

                reasons = []

                if is_negative:
                    reasons.append("Negative depth")
                if is_zero:
                    reasons.append("Zero depth")
                if is_over_100:
                    reasons.append("Depth greater than 100 m")
                if is_deeper_than_well:
                    reasons.append("Measurement deeper than total well depth")

                # Only numeric observations can trigger numeric-depth flags.
                # These flags do not remove or alter the measurement.
                any_review_flag = bool(reasons)

                record = {
                    "Source_file": filename,
                    "Source_sheet": sheet_name,
                    "Source_row_number": int(
                        df.at[idx, "Source_row_number"]
                    ),
                    "Source_district_or_file": district_or_source,
                    "Well_ID_source": source_well_id.loc[idx],
                    "Block": (
                        df.at[idx, find_column(df, ["Block"])]
                        if find_column(df, ["Block"]) is not None
                        else np.nan
                    ),
                    "Year": wl_info["year"],
                    "Season": wl_info["season"],
                    "Original_WL_Column": str(wl_col),
                    "WL_raw": raw_value,
                    "WL_bgl": numeric_value,
                    "WL_status": status,
                    "WL_raw_has_asterisk": has_asterisk,
                    "Total_Depth_column": total_depth_col,
                    "Total_Depth_raw": (
                        df.at[idx, total_depth_col]
                        if total_depth_col is not None
                        else np.nan
                    ),
                    "Total_Depth_numeric": total_depth,
                    "Flag_negative": is_negative,
                    "Flag_zero": is_zero,
                    "Flag_over_100m": is_over_100,
                    "Flag_deeper_than_total_depth": is_deeper_than_well,
                    "Any_review_flag": any_review_flag,
                    "Review_reason": "; ".join(reasons)
                }

                all_long.append(record)

                # Review records retain every flagged measurement and
                # every nonnumeric nonblank measurement for classification.
                if any_review_flag or status == "No reading":
                    review_records.append(record.copy())

        # Preserve original wide measurements in the derived wide output.
        # Do not replace raw WL columns with converted values.
        all_wide.append(df)

        file_reports.append({
            "Source_file": filename,
            "Sheet": sheet_name,
            "Rows": len(df),
            "Water_level_columns": len(wl_columns),
            "Status": "PROCESSED",
            "Details": (
                f"Total-depth column: {total_depth_col}"
                if total_depth_col else
                "Total-depth column not found; well-depth comparison unavailable."
            )
        })

# -------------------- CREATE DATAFRAMES --------------------

wide_df = (
    pd.concat(all_wide, ignore_index=True, sort=False)
    if all_wide else pd.DataFrame()
)

long_df = pd.DataFrame(all_long)
file_report_df = pd.DataFrame(file_reports)
column_report_df = pd.DataFrame(column_reports)
review_df = pd.DataFrame(review_records)

if long_df.empty:
    raise RuntimeError(
        "No seasonal groundwater records were generated. "
        "Review the file report and water-level column naming."
    )

# -------------------- QUALITY-CONTROL SUMMARIES --------------------

status_counts = (
    long_df["WL_status"]
    .value_counts(dropna=False)
    .rename_axis("WL_status")
    .reset_index(name="Measurement_count")
)

reason_counts = []

for flag_col, label in [
    ("Flag_negative", "Negative depth"),
    ("Flag_zero", "Zero depth"),
    ("Flag_over_100m", "Depth greater than 100 m"),
    ("Flag_deeper_than_total_depth", "Deeper than total well depth")
]:
    reason_counts.append({
        "Review_flag": label,
        "Flagged_measurements": int(long_df[flag_col].sum())
    })

flag_summary_df = pd.DataFrame(reason_counts)

# Missing/no-reading values are not treated as numeric errors.
no_reading_df = long_df[
    long_df["WL_status"] == "No reading"
].copy()

dry_df = long_df[
    long_df["WL_status"] == "Dry"
].copy()

numeric_df = long_df[
    long_df["WL_status"] == "Numeric reading"
].copy()

# Summary by year and season.
seasonal_summary_df = (
    long_df.groupby(["Year", "Season", "WL_status"], dropna=False)
    .size()
    .reset_index(name="Measurement_count")
)

# Count source records, distinct wells, and duplicate IDs where possible.
well_id_series = long_df["Well_ID_source"].astype("string")
nonblank_ids = well_id_series[
    well_id_series.notna() &
    (well_id_series.str.strip() != "")
]

qc_summary = pd.DataFrame([{
    "Excel_files_discovered": len(excel_files),
    "Source_sheets_processed": int(
        (file_report_df["Status"] == "PROCESSED").sum()
    ),
    "Source_sheets_not_processed": int(
        (file_report_df["Status"] != "PROCESSED").sum()
    ),
    "Source_wide_rows": len(wide_df),
    "Seasonal_measurement_records": len(long_df),
    "Numeric_readings": len(numeric_df),
    "Dry_observations": len(dry_df),
    "No_reading_observations": len(no_reading_df),
    "Measurements_with_asterisk": int(
        long_df["WL_raw_has_asterisk"].sum()
    ),
    "Negative_depth_measurements": int(
        long_df["Flag_negative"].sum()
    ),
    "Zero_depth_measurements": int(
        long_df["Flag_zero"].sum()
    ),
    "Values_over_100m": int(
        long_df["Flag_over_100m"].sum()
    ),
    "Deeper_than_total_well_depth": int(
        long_df["Flag_deeper_than_total_depth"].sum()
    ),
    "Distinct_nonblank_well_ID_values": int(
        nonblank_ids.nunique()
    ),
    "Missing_or_unconfirmed_ID_values": int(
        len(long_df) - len(nonblank_ids)
    ),
    "Review_records_including_no_reading": len(review_df)
}])

# -------------------- SAVE OUTPUTS --------------------

outputs = {
    "groundwater_cleaned_wide_source_preserved.csv": wide_df,
    "groundwater_cleaned_long_seasonal.csv": long_df,
    "groundwater_measurement_review.csv": review_df,
    "groundwater_no_reading_observations.csv": no_reading_df,
    "groundwater_dry_observations.csv": dry_df,
    "groundwater_numeric_readings.csv": numeric_df,
    "groundwater_status_summary.csv": status_counts,
    "groundwater_review_flag_summary.csv": flag_summary_df,
    "groundwater_seasonal_status_summary.csv": seasonal_summary_df,
    "groundwater_processing_file_report.csv": file_report_df,
    "groundwater_water_level_column_report.csv": column_report_df,
    "groundwater_cleaning_QC_summary.csv": qc_summary
}

for output_name, output_df in outputs.items():
    output_path = os.path.join(OUTPUT_DIR, output_name)
    output_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_path}")

# -------------------- HUMAN-READABLE REPORT --------------------

report_path = os.path.join(
    OUTPUT_DIR,
    "groundwater_cleaning_processing_report.txt"
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write("GROUNDWATER CLEANING PROCESSING REPORT\n")
    f.write("=" * 55 + "\n\n")
    f.write(f"Input directory: {INPUT_DIR}\n")
    f.write(f"Output directory: {OUTPUT_DIR}\n")
    f.write(f"Excel files discovered: {len(excel_files)}\n\n")

    f.write("CLEANING RULES APPLIED\n")
    f.write("-" * 30 + "\n")
    f.write("Dry: classified separately; not converted to numeric.\n")
    f.write("Numeric readings: retained, including zero and negative values.\n")
    f.write("Values such as 13.10*: numeric portion extracted for analysis.\n")
    f.write("Blank and unconfirmed nonnumeric values: No reading.\n")
    f.write("Values >100 m: retained and flagged.\n")
    f.write("Values deeper than total well depth: retained and flagged.\n")
    f.write("Original Excel files: not modified.\n")
    f.write("Coordinates: not automatically corrected.\n\n")

    f.write("QUALITY-CONTROL SUMMARY\n")
    f.write("-" * 30 + "\n")
    f.write(qc_summary.to_string(index=False))
    f.write("\n\nMEASUREMENT STATUS COUNTS\n")
    f.write(status_counts.to_string(index=False))
    f.write("\n\nREVIEW FLAG COUNTS\n")
    f.write(flag_summary_df.to_string(index=False))
    f.write("\n\nFILE PROCESSING REPORT\n")
    f.write(file_report_df.to_string(index=False))
    f.write("\n")

print("\n" + "=" * 65)
print("GROUNDWATER CLEANING FINISHED")
print("=" * 65)
print(qc_summary.to_string(index=False))
print("\nMeasurement status counts:")
print(status_counts.to_string(index=False))
print("\nReview flags:")
print(flag_summary_df.to_string(index=False))
print(f"\nAll outputs saved in:\n{OUTPUT_DIR}")
print("\nReview the file-processing report for unprocessed sheets.")


**[orig #11]** Well elevation and estimated head audit  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# GROUNDWATER WELL ELEVATION + ESTIMATED HEAD AUDIT
# Project: Jaipur / Rajasthan groundwater research
#
# Reads original Excel files WITHOUT modifying them.
# Retrieves terrain elevation from SRTM 90m via OpenTopoData.
# Calculates estimated groundwater head:
#     head elevation = DEM ground elevation - depth below ground
#
# IMPORTANT:
# - DEM elevations are estimates, not surveyed well elevations.
# - Verify the elevation datum before combining with other datasets.
# - Head estimates are not suitable for final calibration without QC.
# - Dry and no-reading observations are not converted to head.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------- PATHS -----------------------

INPUT_DIR = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

OUTPUT_DIR = INPUT_DIR / "Well_Elevation_and_Head_Estimates"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------- SETTINGS -----------------------

# Public elevation service. SRTM 90 m is a regional terrain estimate.
ELEVATION_API = "https://api.opentopodata.org/v1/srtm90m"

# Public API services may limit request size/rate.
BATCH_SIZE = 80
REQUEST_PAUSE_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 90
MAX_RETRIES = 4

# Coordinate bounds for a broad Rajasthan plausibility check.
RAJASTHAN_LAT_MIN, RAJASTHAN_LAT_MAX = 23.0, 31.5
RAJASTHAN_LON_MIN, RAJASTHAN_LON_MAX = 68.0, 79.5

# Candidate coordinate and ID column names.
LAT_CANDIDATES = [
    "Latitude", "Lat", "Latitude_DD", "Latitude_decimal",
    "Well_Latitude", "Lat_DD", "Y_coordinate"
]
LON_CANDIDATES = [
    "Longitude", "Lon", "Long", "Longitude_DD",
    "Longitude_decimal", "Well_Longitude", "Long_DD", "X_coordinate"
]
ID_CANDIDATES = [
    "Well_ID", "WellID", "Well ID", "Unique_ID", "UniqueID",
    "ID", "Well_No", "Well Number", "Monitoring_Well_ID"
]
TOTAL_DEPTH_CANDIDATES = [
    "Total_Depth_bgl", "Total_Depth", "Total depth",
    "Total_Depth_BGL", "Well_Depth"
]

# ----------------------- HELPERS -----------------------

def norm(value):
    return re.sub(r"[^a-z0-9]", "", str(value).strip().lower())


def find_col(columns, candidates):
    lookup = {norm(c): c for c in columns}
    for candidate in candidates:
        key = norm(candidate)
        if key in lookup:
            return lookup[key]
    return None


def find_wl_columns(columns):
    found = []
    for col in columns:
        match = re.fullmatch(
            r"WL[_\s-]*(pre|post)[_\s-]*(\d{2}|\d{4})",
            str(col).strip(),
            flags=re.IGNORECASE
        )
        if match:
            season = match.group(1).capitalize()
            year_text = match.group(2)
            year = int(year_text)
            if len(year_text) == 2:
                year += 2000
            found.append((col, year, season))
    return found


def numeric_measurement(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()

    if not text or text.lower() == "dry":
        return np.nan

    # Accept values such as 13.10*; use 13.10 numerically.
    text = text.replace(",", "")
    match = re.fullmatch(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*\*?",
        text
    )

    if not match:
        return np.nan

    try:
        return float(match.group(1))
    except ValueError:
        return np.nan


def classify_raw(value):
    if pd.isna(value):
        return "No reading"

    text = str(value).strip()

    if text == "":
        return "No reading"

    if text.lower() == "dry":
        return "Dry"

    if pd.notna(numeric_measurement(value)):
        return "Numeric reading"

    return "No reading"


def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return False

    # Flag likely swapped coordinates or coordinates outside Rajasthan.
    if not (
        RAJASTHAN_LAT_MIN <= lat <= RAJASTHAN_LAT_MAX
        and RAJASTHAN_LON_MIN <= lon <= RAJASTHAN_LON_MAX
    ):
        return False

    return True


def query_elevations(coordinates):
    """
    coordinates: list of (latitude, longitude)
    Returns one record per input coordinate, retaining order.
    """
    if not coordinates:
        return []

    location_string = "|".join(
        f"{lat:.7f},{lon:.7f}" for lat, lon in coordinates
    )

    params = {"locations": location_string}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(
                ELEVATION_API,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
                headers={
                    "User-Agent": "Groundwater-PhD-DEM-Audit/1.0"
                }
            )

            response.raise_for_status()
            payload = response.json()

            results = payload.get("results", [])

            if len(results) != len(coordinates):
                raise ValueError(
                    f"Expected {len(coordinates)} results; "
                    f"received {len(results)}."
                )

            records = []

            for coord, result in zip(coordinates, results):
                elevation = result.get("elevation")

                records.append({
                    "Latitude_query": coord[0],
                    "Longitude_query": coord[1],
                    "DEM_ground_elevation_m": (
                        float(elevation)
                        if elevation is not None
                        else np.nan
                    ),
                    "Elevation_source": "OpenTopoData SRTM 90m",
                    "Elevation_status": (
                        "OK" if elevation is not None else "NO_ELEVATION"
                    )
                })

            return records

        except Exception as exc:
            print(
                f"Elevation request attempt {attempt}/{MAX_RETRIES} "
                f"failed: {exc}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(2 * attempt)

    return [
        {
            "Latitude_query": lat,
            "Longitude_query": lon,
            "DEM_ground_elevation_m": np.nan,
            "Elevation_source": "OpenTopoData SRTM 90m",
            "Elevation_status": "API_FAILED"
        }
        for lat, lon in coordinates
    ]


# ----------------------- READ ORIGINAL EXCEL FILES -----------------------

excel_files = sorted(
    list(INPUT_DIR.glob("*.xlsx")) +
    list(INPUT_DIR.glob("*.xls"))
)

excel_files = [
    p for p in excel_files
    if not p.name.startswith("~$")
]

if not excel_files:
    raise FileNotFoundError(
        f"No original Excel workbooks found in {INPUT_DIR}"
    )

well_records = []
measurement_records = []
file_issues = []

for file_path in excel_files:
    print("Reading:", file_path.name)

    try:
        workbook = pd.ExcelFile(file_path)
    except Exception as exc:
        file_issues.append({
            "Source_file": file_path.name,
            "Issue": "Workbook read error",
            "Details": str(exc)
        })
        continue

    for sheet_name in workbook.sheet_names:
        try:
            df = pd.read_excel(
                file_path,
                sheet_name=sheet_name,
                dtype=object
            )
        except Exception as exc:
            file_issues.append({
                "Source_file": file_path.name,
                "Issue": "Sheet read error",
                "Details": str(exc)
            })
            continue

        if df.empty:
            continue

        lat_col = find_col(df.columns, LAT_CANDIDATES)
        lon_col = find_col(df.columns, LON_CANDIDATES)
        id_col = find_col(df.columns, ID_CANDIDATES)
        total_depth_col = find_col(
            df.columns, TOTAL_DEPTH_CANDIDATES
        )
        block_col = find_col(df.columns, ["Block"])
        district_col = find_col(
            df.columns, ["District", "District_Name"]
        )

        wl_cols = find_wl_columns(df.columns)

        if lat_col is None or lon_col is None:
            file_issues.append({
                "Source_file": file_path.name,
                "Issue": "Coordinate columns not found",
                "Details": (
                    f"Latitude column={lat_col}; longitude column={lon_col}; "
                    f"sheet={sheet_name}"
                )
            })

        df = df.copy()
        df["_source_row"] = np.arange(2, len(df) + 2)

        for idx, row in df.iterrows():
            raw_lat = row[lat_col] if lat_col else np.nan
            raw_lon = row[lon_col] if lon_col else np.nan

            lat = pd.to_numeric(
                pd.Series([raw_lat]), errors="coerce"
            ).iloc[0]
            lon = pd.to_numeric(
                pd.Series([raw_lon]), errors="coerce"
            ).iloc[0]

            if id_col:
                raw_id = row[id_col]
                well_id = (
                    str(raw_id).strip()
                    if pd.notna(raw_id) and str(raw_id).strip()
                    else f"{file_path.stem}::{sheet_name}::row{row['_source_row']}"
                )
            else:
                well_id = (
                    f"{file_path.stem}::{sheet_name}::row{row['_source_row']}"
                )

            total_depth_raw = (
                row[total_depth_col]
                if total_depth_col else np.nan
            )
            total_depth_numeric = numeric_measurement(total_depth_raw)

            block = row[block_col] if block_col else np.nan
            district = row[district_col] if district_col else file_path.stem

            well_record = {
                "Well_key": f"{file_path.name}|{sheet_name}|{row['_source_row']}",
                "Well_ID_source": well_id,
                "District": district,
                "Block": block,
                "Source_file": file_path.name,
                "Source_sheet": sheet_name,
                "Source_row_number": int(row["_source_row"]),
                "Latitude_raw": raw_lat,
                "Longitude_raw": raw_lon,
                "Latitude": lat,
                "Longitude": lon,
                "Coordinate_valid_Rajasthan_bounds": valid_coordinate(lat, lon),
                "Total_Depth_raw": total_depth_raw,
                "Total_Depth_numeric_m": total_depth_numeric,
            }

            well_records.append(well_record)

            for wl_col, year, season in wl_cols:
                raw_wl = row[wl_col]
                status = classify_raw(raw_wl)
                depth = numeric_measurement(raw_wl)

                measurement_records.append({
                    "Well_key": well_record["Well_key"],
                    "Well_ID_source": well_id,
                    "District": district,
                    "Block": block,
                    "Source_file": file_path.name,
                    "Source_sheet": sheet_name,
                    "Source_row_number": int(row["_source_row"]),
                    "Year": year,
                    "Season": season,
                    "Original_WL_Column": str(wl_col),
                    "WL_raw": raw_wl,
                    "WL_status": status,
                    "Depth_to_water_bgl_m": depth
                })

wells = pd.DataFrame(well_records)
measurements = pd.DataFrame(measurement_records)

if wells.empty:
    raise RuntimeError("No well rows were read.")

if measurements.empty:
    raise RuntimeError(
        "No water-level columns were recognized. Check the source headers."
    )

# ----------------------- UNIQUE COORDINATE LOOKUP -----------------------

valid_wells = wells[
    wells["Coordinate_valid_Rajasthan_bounds"]
].copy()

# Query each unique coordinate only once.
unique_coords = (
    valid_wells[["Latitude", "Longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nSource well rows:", len(wells))
print("Unique valid coordinates:", len(unique_coords))
print("Coordinate-invalid/missing rows:", len(wells) - len(valid_wells))

elevation_lookup_records = []

coords_list = list(
    zip(unique_coords["Latitude"], unique_coords["Longitude"])
)

for start in range(0, len(coords_list), BATCH_SIZE):
    batch = coords_list[start:start + BATCH_SIZE]
    print(
        f"Elevation query batch {start // BATCH_SIZE + 1}: "
        f"{len(batch)} coordinates"
    )

    batch_results = query_elevations(batch)
    elevation_lookup_records.extend(batch_results)

    if start + BATCH_SIZE < len(coords_list):
        time.sleep(REQUEST_PAUSE_SECONDS)

elevation_lookup = pd.DataFrame(elevation_lookup_records)

# Match elevation results to coordinate keys.
if not elevation_lookup.empty:
    elevation_lookup["coord_key"] = list(zip(
        elevation_lookup["Latitude_query"].round(7),
        elevation_lookup["Longitude_query"].round(7)
    ))

wells["coord_key"] = list(zip(
    pd.to_numeric(wells["Latitude"], errors="coerce").round(7),
    pd.to_numeric(wells["Longitude"], errors="coerce").round(7)
))

if not elevation_lookup.empty:
    wells = wells.merge(
        elevation_lookup[
            [
                "coord_key",
                "DEM_ground_elevation_m",
                "Elevation_source",
                "Elevation_status"
            ]
        ],
        on="coord_key",
        how="left"
    )
else:
    wells["DEM_ground_elevation_m"] = np.nan
    wells["Elevation_source"] = "OpenTopoData SRTM 90m"
    wells["Elevation_status"] = "API_FAILED"

# Coordinates invalid or missing cannot be queried.
wells.loc[
    ~wells["Coordinate_valid_Rajasthan_bounds"],
    "Elevation_status"
] = "INVALID_OR_MISSING_COORDINATES"

# Flag suspiciously absent elevations.
wells["Elevation_review_flag"] = np.where(
    wells["DEM_ground_elevation_m"].isna(),
    "Elevation unavailable; investigate coordinates/API",
    "DEM estimate; validate before calibration"
)

# ----------------------- CALCULATE ESTIMATED HEAD -----------------------

measurements = measurements.merge(
    wells[
        [
            "Well_key",
            "Latitude",
            "Longitude",
            "DEM_ground_elevation_m",
            "Elevation_source",
            "Elevation_status",
            "Elevation_review_flag",
            "Coordinate_valid_Rajasthan_bounds"
        ]
    ],
    on="Well_key",
    how="left"
)

measurements["Estimated_head_m_AMSL"] = (
    measurements["DEM_ground_elevation_m"]
    - measurements["Depth_to_water_bgl_m"]
)

# Only numeric observations with available elevations get a head estimate.
measurements.loc[
    (measurements["WL_status"] != "Numeric reading") |
    measurements["DEM_ground_elevation_m"].isna(),
    "Estimated_head_m_AMSL"
] = np.nan

measurements["Head_estimate_quality"] = np.select(
    [
        measurements["WL_status"] == "Dry",
        measurements["WL_status"] == "No reading",
        measurements["DEM_ground_elevation_m"].isna(),
        measurements["WL_status"] == "Numeric reading"
    ],
    [
        "Not calculated: dry well",
        "Not calculated: no reading",
        "Not calculated: elevation unavailable",
        "Provisional: DEM-derived ground elevation"
    ],
    default="Review"
)

# ----------------------- SAVE RESULTS -----------------------

well_output = OUTPUT_DIR / "well_inventory_with_dem_elevation.csv"
measurement_output = OUTPUT_DIR / "groundwater_measurements_with_estimated_head.csv"
missing_coord_output = OUTPUT_DIR / "wells_missing_or_invalid_coordinates.csv"
missing_elev_output = OUTPUT_DIR / "wells_without_dem_elevation.csv"
issue_output = OUTPUT_DIR / "source_file_and_column_issues.csv"
summary_output = OUTPUT_DIR / "elevation_head_processing_summary.csv"

wells.drop(columns=["coord_key"], errors="ignore").to_csv(
    well_output, index=False, encoding="utf-8-sig"
)

measurements.to_csv(
    measurement_output, index=False, encoding="utf-8-sig"
)

wells[
    ~wells["Coordinate_valid_Rajasthan_bounds"]
].to_csv(
    missing_coord_output, index=False, encoding="utf-8-sig"
)

wells[
    wells["DEM_ground_elevation_m"].isna()
].to_csv(
    missing_elev_output, index=False, encoding="utf-8-sig"
)

pd.DataFrame(file_issues).to_csv(
    issue_output, index=False, encoding="utf-8-sig"
)

summary = pd.DataFrame([{
    "Source_files_found": len(excel_files),
    "Source_well_rows_read": len(wells),
    "Unique_well_ID_values": wells["Well_ID_source"].nunique(),
    "Unique_coordinates_queried": len(unique_coords),
    "Valid_coordinate_rows": int(
        wells["Coordinate_valid_Rajasthan_bounds"].sum()
    ),
    "Rows_with_DEM_elevation": int(
        wells["DEM_ground_elevation_m"].notna().sum()
    ),
    "Rows_without_DEM_elevation": int(
        wells["DEM_ground_elevation_m"].isna().sum()
    ),
    "Seasonal_measurement_records": len(measurements),
    "Numeric_depth_measurements": int(
        measurements["WL_status"].eq("Numeric reading").sum()
    ),
    "Dry_observations": int(
        measurements["WL_status"].eq("Dry").sum()
    ),
    "No_reading_observations": int(
        measurements["WL_status"].eq("No reading").sum()
    ),
    "Estimated_heads_calculated": int(
        measurements["Estimated_head_m_AMSL"].notna().sum()
    ),
    "Source_file_or_column_issues": len(file_issues),
    "Elevation_service": "OpenTopoData SRTM 90m",
}])

summary.to_csv(summary_output, index=False, encoding="utf-8-sig")

print("\n" + "=" * 65)
print("ELEVATION AND HEAD ESTIMATION FINISHED")
print("=" * 65)
print(summary.to_string(index=False))

print("\nSaved outputs:")
for path in [
    well_output,
    measurement_output,
    missing_coord_output,
    missing_elev_output,
    issue_output,
    summary_output
]:
    print(path)

print("\nIMPORTANT:")
print(
    "Estimated_head_m_AMSL is provisional. Verify the DEM vertical "
    "datum, coordinate accuracy, and ground elevations before "
    "using the values in MODFLOW calibration."
)


**[orig #12]** Reconcile cleaned outputs with elevation/head outputs  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# GROUNDWATER RECONCILIATION AUDIT
# Compare cleaned groundwater outputs with elevation/head outputs.
# Read-only: does not modify original workbooks or existing outputs.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime

# ------------------------- 1. PATHS -------------------------------

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)

CLEAN_DIR = BASE / "Groundwater_Cleaning_According_to_User_Rules"
ELEV_DIR = BASE / "Well_Elevation_and_Head_Estimates"

AUDIT_DIR = BASE / "Groundwater_Reconciliation_Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_WIDE = CLEAN_DIR / "groundwater_cleaned_wide_source_preserved.csv"
CLEAN_LONG = CLEAN_DIR / "groundwater_cleaned_long_seasonal.csv"

ELEV_INVENTORY = ELEV_DIR / "well_inventory_with_dem_elevation.csv"
ELEV_MEASUREMENTS = ELEV_DIR / "groundwater_measurements_with_estimated_head.csv"

REQUIRED_FILES = [
    CLEAN_WIDE,
    CLEAN_LONG,
    ELEV_INVENTORY,
    ELEV_MEASUREMENTS,
]

missing_files = [str(p) for p in REQUIRED_FILES if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "The following expected files were not found:\n"
        + "\n".join(missing_files)
        + "\nCheck the folder paths before running the audit."
    )

# ------------------------- 2. LOAD FILES --------------------------

def read_csv_safely(path):
    """Read a CSV while preserving identifiers and source values as text."""
    df = pd.read_csv(path, low_memory=False, dtype=str)
    df.columns = [str(c).strip() for c in df.columns]
    return df

clean_wide = read_csv_safely(CLEAN_WIDE)
clean_long = read_csv_safely(CLEAN_LONG)
elev_inventory = read_csv_safely(ELEV_INVENTORY)
elev_measurements = read_csv_safely(ELEV_MEASUREMENTS)

print("=" * 72)
print("GROUNDWATER RECONCILIATION AUDIT")
print("=" * 72)
print(f"Cleaned wide rows:            {len(clean_wide):,}")
print(f"Cleaned seasonal rows:        {len(clean_long):,}")
print(f"Elevation inventory rows:     {len(elev_inventory):,}")
print(f"Elevation measurement rows:   {len(elev_measurements):,}")
print(f"Audit output folder:          {AUDIT_DIR}")
print()

# ------------------------- 3. COLUMN UTILITIES -------------------

def norm_col(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

def find_col(df, candidates, required=False, label="column"):
    """Find a column by case/punctuation-insensitive candidate names."""
    lookup = {norm_col(c): c for c in df.columns}

    for candidate in candidates:
        key = norm_col(candidate)
        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            f"Could not identify {label}. Candidates: {candidates}\n"
            f"Available columns: {list(df.columns)}"
        )
    return None

def find_source_col(df):
    return find_col(
        df,
        ["Source_file", "source_filename", "source_workbook",
         "file_name", "filename", "sourcefile"],
        required=True,
        label="source file column"
    )

def find_sheet_col(df):
    return find_col(
        df,
        ["Source_sheet", "source_worksheet", "sheet_name",
         "worksheet", "sourcesheet"],
        required=False
    )

def find_row_col(df):
    return find_col(
        df,
        ["Source_row_number", "source_row", "original_row_number",
         "excel_row_number", "row_number", "sourcerownumber"],
        required=True,
        label="original source row number"
    )

def find_id_col(df):
    return find_col(
        df,
        ["Well_ID", "Well ID", "wellid", "Unique_Well_ID",
         "Unique Well ID", "well_code", "Well_Code",
         "Monitoring_Well_ID", "ID"],
        required=False
    )

def find_lat_col(df):
    return find_col(
        df,
        ["Latitude", "Lat", "latitude_decimal", "latitude_dd",
         "Latitude_DD", "Y_coordinate"],
        required=False
    )

def find_lon_col(df):
    return find_col(
        df,
        ["Longitude", "Long", "Lon", "longitude_decimal",
         "longitude_dd", "Longitude_DD", "X_coordinate"],
        required=False
    )

def normalize_file(value):
    """Normalize paths to workbook basenames for cross-output comparison."""
    if pd.isna(value):
        return ""
    value = str(value).strip().replace("\\", "/")
    return value.split("/")[-1].casefold()

def normalize_sheet(value):
    if pd.isna(value):
        return ""
    return str(value).strip().casefold()

def normalize_row(value):
    """Convert source row values to a comparable representation."""
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(str(value).strip())))
    except Exception:
        return str(value).strip()

def normalize_id(value):
    """Normalize IDs for comparison without changing the source files."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.casefold() in {"", "nan", "none", "null", "na", "n/a"}:
        return ""
    return re.sub(r"\s+", " ", text).casefold()

def numeric_coord(series):
    return pd.to_numeric(
        series.astype(str).str.strip().replace(
            {"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan}
        ),
        errors="coerce"
    )

# ------------------------- 4. CREATE SOURCE-ROW KEYS --------------

def add_source_keys(df, label):
    source_col = find_source_col(df)
    row_col = find_row_col(df)
    sheet_col = find_sheet_col(df)

    out = df.copy()
    out["_audit_file"] = out[source_col].map(normalize_file)
    out["_audit_sheet"] = (
        out[sheet_col].map(normalize_sheet)
        if sheet_col else ""
    )
    out["_audit_row"] = out[row_col].map(normalize_row)

    # Keep blank/invalid keys visible; don't silently treat them as valid.
    out["_audit_key_valid"] = (
        out["_audit_file"].ne("")
        & out["_audit_row"].ne("")
    )

    out["_audit_source_key"] = (
        out["_audit_file"] + "||"
        + out["_audit_sheet"] + "||"
        + out["_audit_row"]
    )

    invalid = out.loc[~out["_audit_key_valid"]].copy()
    if not invalid.empty:
        invalid.to_csv(
            AUDIT_DIR / f"{label}_invalid_source_keys.csv",
            index=False
        )

    return out

cw = add_source_keys(clean_wide, "cleaned_wide")
cl = add_source_keys(clean_long, "cleaned_long")
ei = add_source_keys(elev_inventory, "elevation_inventory")
em = add_source_keys(elev_measurements, "elevation_measurements")

# ------------------------- 5. KEY UNIQUENESS ----------------------

def key_summary(df, label):
    valid = df.loc[df["_audit_key_valid"]]
    duplicate_mask = valid.duplicated("_audit_source_key", keep=False)

    report = {
        "dataset": label,
        "rows_total": len(df),
        "rows_with_valid_source_key": int(df["_audit_key_valid"].sum()),
        "rows_with_invalid_source_key": int((~df["_audit_key_valid"]).sum()),
        "distinct_source_keys": int(valid["_audit_source_key"].nunique()),
        "duplicate_source_key_rows": int(duplicate_mask.sum()),
        "duplicate_source_keys": int(
            valid.loc[duplicate_mask, "_audit_source_key"].nunique()
        ),
    }

    if duplicate_mask.any():
        valid.loc[duplicate_mask].sort_values(
            "_audit_source_key"
        ).to_csv(
            AUDIT_DIR / f"{label}_duplicate_source_keys.csv",
            index=False
        )

    return report

key_reports = pd.DataFrame([
    key_summary(cw, "cleaned_wide"),
    key_summary(cl, "cleaned_long"),
    key_summary(ei, "elevation_inventory"),
    key_summary(em, "elevation_measurements"),
])
key_reports.to_csv(AUDIT_DIR / "source_key_quality_summary.csv", index=False)

# ------------------------- 6. COMPARE SOURCE ROWS -----------------

def unique_key_table(df):
    valid = df.loc[df["_audit_key_valid"]]
    return valid.drop_duplicates("_audit_source_key")[
        ["_audit_source_key", "_audit_file", "_audit_sheet", "_audit_row"]
    ].copy()

wide_keys = unique_key_table(cw)
elev_keys = unique_key_table(ei)

wide_set = set(wide_keys["_audit_source_key"])
elev_set = set(elev_keys["_audit_source_key"])

only_elevation = sorted(elev_set - wide_set)
only_cleaned = sorted(wide_set - elev_set)
common_keys = sorted(wide_set & elev_set)

def rows_for_keys(df, keys):
    if not keys:
        return df.iloc[0:0].copy()
    return df[df["_audit_source_key"].isin(keys)].copy()

rows_only_elevation = rows_for_keys(ei, only_elevation)
rows_only_cleaned = rows_for_keys(cw, only_cleaned)

rows_only_elevation.to_csv(
    AUDIT_DIR / "source_rows_present_only_in_elevation_inventory.csv",
    index=False
)
rows_only_cleaned.to_csv(
    AUDIT_DIR / "source_rows_present_only_in_cleaned_wide.csv",
    index=False
)

# Compare source-row counts by workbook and worksheet.
def count_by_file_sheet(df, name):
    valid = df.loc[df["_audit_key_valid"]]
    return (
        valid.groupby(["_audit_file", "_audit_sheet"], dropna=False)
        .agg(
            rows_in_output=("_audit_source_key", "size"),
            distinct_source_rows=("_audit_source_key", "nunique")
        )
        .reset_index()
        .rename(columns={
            "rows_in_output": f"{name}_rows_in_output",
            "distinct_source_rows": f"{name}_distinct_source_rows"
        })
    )

wide_counts = count_by_file_sheet(cw, "cleaned_wide")
elev_counts = count_by_file_sheet(ei, "elevation_inventory")

file_sheet_comparison = wide_counts.merge(
    elev_counts,
    on=["_audit_file", "_audit_sheet"],
    how="outer"
).fillna(0)

for col in file_sheet_comparison.columns:
    if col.endswith("_rows_in_output") or col.endswith("_distinct_source_rows"):
        file_sheet_comparison[col] = file_sheet_comparison[col].astype(int)

if {"cleaned_wide_distinct_source_rows",
    "elevation_inventory_distinct_source_rows"}.issubset(file_sheet_comparison.columns):
    file_sheet_comparison["elevation_minus_cleaned_distinct_rows"] = (
        file_sheet_comparison["elevation_inventory_distinct_source_rows"]
        - file_sheet_comparison["cleaned_wide_distinct_source_rows"]
    )

file_sheet_comparison.to_csv(
    AUDIT_DIR / "source_row_counts_by_file_and_sheet.csv",
    index=False
)

# ------------------------- 7. SEASONAL RECORD COUNTS --------------

def seasonal_count_by_key(df, label):
    valid = df.loc[df["_audit_key_valid"]]
    result = (
        valid.groupby("_audit_source_key", dropna=False)
        .size()
        .rename(f"{label}_seasonal_records")
        .reset_index()
    )
    return result

clean_season_counts = seasonal_count_by_key(cl, "cleaned")
elev_season_counts = seasonal_count_by_key(em, "elevation")

season_comparison = clean_season_counts.merge(
    elev_season_counts,
    on="_audit_source_key",
    how="outer"
)

season_comparison[
    ["cleaned_seasonal_records", "elevation_seasonal_records"]
] = season_comparison[
    ["cleaned_seasonal_records", "elevation_seasonal_records"]
].fillna(0).astype(int)

season_comparison["elevation_minus_cleaned_records"] = (
    season_comparison["elevation_seasonal_records"]
    - season_comparison["cleaned_seasonal_records"]
)

# Add readable source-file/sheet/row information.
key_lookup = ei[
    ["_audit_source_key", "_audit_file", "_audit_sheet", "_audit_row"]
].drop_duplicates("_audit_source_key")

season_comparison = season_comparison.merge(
    key_lookup, on="_audit_source_key", how="left"
)

season_comparison.to_csv(
    AUDIT_DIR / "seasonal_record_counts_by_source_row.csv",
    index=False
)

season_discrepancies = season_comparison[
    season_comparison["elevation_minus_cleaned_records"] != 0
].copy()

season_discrepancies.to_csv(
    AUDIT_DIR / "seasonal_record_discrepancies.csv",
    index=False
)

# ------------------------- 8. CHECK MEASUREMENT COLUMN COUNTS ------

def find_measurement_col(df):
    return find_col(
        df,
        ["WL_column", "water_level_column", "source_WL_column",
         "measurement_column", "season_column", "period_column",
         "WL_variable", "observation_column"],
        required=False
    )

clean_measure_col = find_measurement_col(cl)
elev_measure_col = find_measurement_col(em)

measurement_column_comparison = None

if clean_measure_col and elev_measure_col:
    clean_counts = (
        cl.groupby(clean_measure_col, dropna=False)
        .size().rename("cleaned_record_count").reset_index()
        .rename(columns={clean_measure_col: "measurement_column"})
    )
    elev_counts_m = (
        em.groupby(elev_measure_col, dropna=False)
        .size().rename("elevation_record_count").reset_index()
        .rename(columns={elev_measure_col: "measurement_column"})
    )

    measurement_column_comparison = clean_counts.merge(
        elev_counts_m, on="measurement_column", how="outer"
    ).fillna(0)

    measurement_column_comparison["cleaned_record_count"] = (
        measurement_column_comparison["cleaned_record_count"].astype(int)
    )
    measurement_column_comparison["elevation_record_count"] = (
        measurement_column_comparison["elevation_record_count"].astype(int)
    )
    measurement_column_comparison["elevation_minus_cleaned"] = (
        measurement_column_comparison["elevation_record_count"]
        - measurement_column_comparison["cleaned_record_count"]
    )

    measurement_column_comparison.to_csv(
        AUDIT_DIR / "seasonal_counts_by_measurement_column.csv",
        index=False
    )

# ------------------------- 9. DUPLICATE WELL RECORDS ---------------

def attach_identity_coordinates(df, label):
    out = df.copy()
    id_col = find_id_col(out)
    lat_col = find_lat_col(out)
    lon_col = find_lon_col(out)

    out["_audit_id_column_found"] = id_col or ""
    out["_audit_lat_column_found"] = lat_col or ""
    out["_audit_lon_column_found"] = lon_col or ""

    if id_col:
        out["_audit_well_id"] = out[id_col].map(normalize_id)
    else:
        out["_audit_well_id"] = ""

    if lat_col:
        out["_audit_lat"] = numeric_coord(out[lat_col])
    else:
        out["_audit_lat"] = np.nan

    if lon_col:
        out["_audit_lon"] = numeric_coord(out[lon_col])
    else:
        out["_audit_lon"] = np.nan

    out["_audit_coordinates_valid"] = (
        out["_audit_lat"].notna()
        & out["_audit_lon"].notna()
        & out["_audit_lat"].between(-90, 90)
        & out["_audit_lon"].between(-180, 180)
    )

    return out, id_col, lat_col, lon_col

cw_id, cw_id_col, cw_lat_col, cw_lon_col = attach_identity_coordinates(
    cw, "cleaned_wide"
)
ei_id, ei_id_col, ei_lat_col, ei_lon_col = attach_identity_coordinates(
    ei, "elevation_inventory"
)

# Preserve rows with missing ID in the source-row audit, but do not group
# blank IDs together and label them as a duplicate well.
def well_duplicate_report(df, label):
    valid = df[
        df["_audit_well_id"].ne("")
        & df["_audit_coordinates_valid"]
    ].copy()

    if valid.empty:
        return pd.DataFrame(), pd.DataFrame()

    # Coordinate pairs rounded to 6 decimal places for comparison.
    # Original values remain available in the source columns.
    valid["_lat_round6"] = valid["_audit_lat"].round(6)
    valid["_lon_round6"] = valid["_audit_lon"].round(6)

    summary = (
        valid.groupby("_audit_well_id")
        .agg(
            source_rows=("_audit_source_key", "nunique"),
            coordinate_pairs=(
                "_lat_round6",
                lambda x: len(set(zip(
                    x,
                    valid.loc[x.index, "_lon_round6"]
                )))
            ),
            latitude_min=("_audit_lat", "min"),
            latitude_max=("_audit_lat", "max"),
            longitude_min=("_audit_lon", "min"),
            longitude_max=("_audit_lon", "max"),
        )
        .reset_index()
        .rename(columns={"_audit_well_id": "normalized_well_id"})
    )

    summary["same_coordinates_for_id"] = summary["coordinate_pairs"] == 1
    summary["multiple_source_rows_for_id"] = summary["source_rows"] > 1

    summary.to_csv(
        AUDIT_DIR / f"{label}_well_id_coordinate_summary.csv",
        index=False
    )

    repeated = valid[
        valid["_audit_well_id"].isin(
            summary.loc[
                summary["multiple_source_rows_for_id"],
                "normalized_well_id"
            ]
        )
    ].sort_values("_audit_well_id")

    repeated.to_csv(
        AUDIT_DIR / f"{label}_repeated_well_id_records.csv",
        index=False
    )

    inconsistent_ids = set(
        summary.loc[
            ~summary["same_coordinates_for_id"],
            "normalized_well_id"
        ]
    )
    inconsistent = valid[
        valid["_audit_well_id"].isin(inconsistent_ids)
    ].sort_values("_audit_well_id")

    inconsistent.to_csv(
        AUDIT_DIR / f"{label}_well_ids_with_inconsistent_coordinates.csv",
        index=False
    )

    return summary, repeated

wide_id_summary, wide_repeated = well_duplicate_report(
    cw_id, "cleaned_wide"
)
elev_id_summary, elev_repeated = well_duplicate_report(
    ei_id, "elevation_inventory"
)

# ------------------------- 10. ID-TO-COORDINATE CROSS-OUTPUT CHECK -

def id_coordinate_sets(df):
    valid = df[
        df["_audit_well_id"].ne("")
        & df["_audit_coordinates_valid"]
    ].copy()

    valid["_lat_round6"] = valid["_audit_lat"].round(6)
    valid["_lon_round6"] = valid["_audit_lon"].round(6)

    return (
        valid.groupby("_audit_well_id")
        .apply(
            lambda g: sorted(set(zip(
                g["_lat_round6"].tolist(),
                g["_lon_round6"].tolist()
            ))),
            include_groups=False
        )
        .to_dict()
    )

wide_id_coords = id_coordinate_sets(cw_id)
elev_id_coords = id_coordinate_sets(ei_id)

all_ids = sorted(set(wide_id_coords) | set(elev_id_coords))
id_compare_rows = []

for wid in all_ids:
    a = wide_id_coords.get(wid, [])
    b = elev_id_coords.get(wid, [])

    if not a:
        status = "ID has valid coordinates only in elevation inventory"
    elif not b:
        status = "ID has valid coordinates only in cleaned wide"
    elif set(a) == set(b):
        status = "CONSISTENT"
    else:
        status = "COORDINATE SET MISMATCH"

    id_compare_rows.append({
        "normalized_well_id": wid,
        "cleaned_coordinate_pairs": json.dumps(a),
        "elevation_coordinate_pairs": json.dumps(b),
        "cleaned_coordinate_pair_count": len(a),
        "elevation_coordinate_pair_count": len(b),
        "coordinate_mapping_status": status
    })

id_coordinate_comparison = pd.DataFrame(id_compare_rows)
id_coordinate_comparison.to_csv(
    AUDIT_DIR / "well_id_coordinate_mapping_crosscheck.csv",
    index=False
)

# ------------------------- 11. DUPLICATE COORDINATE PAIRS ----------

def duplicate_coordinate_report(df, label):
    valid = df[df["_audit_coordinates_valid"]].copy()
    valid["_lat_round6"] = valid["_audit_lat"].round(6)
    valid["_lon_round6"] = valid["_audit_lon"].round(6)

    counts = (
        valid.groupby(["_lat_round6", "_lon_round6"])
        .agg(
            source_rows=("_audit_source_key", "nunique"),
            well_ids=(
                "_audit_well_id",
                lambda x: len(set(v for v in x if v))
            )
        )
        .reset_index()
    )

    duplicates = counts[
        (counts["source_rows"] > 1) | (counts["well_ids"] > 1)
    ].copy()

    duplicates.to_csv(
        AUDIT_DIR / f"{label}_duplicate_coordinate_pairs.csv",
        index=False
    )
    return duplicates

wide_duplicate_coords = duplicate_coordinate_report(cw_id, "cleaned_wide")
elev_duplicate_coords = duplicate_coordinate_report(ei_id, "elevation_inventory")

# ------------------------- 12. SUMMARY AND INTERPRETATION ----------

def col_exists(df, col):
    return col is not None and col in df.columns

summary = {
    "audit_timestamp_local": datetime.now().isoformat(timespec="seconds"),
    "cleaned_wide_rows": len(clean_wide),
    "cleaned_long_seasonal_rows": len(clean_long),
    "elevation_inventory_rows": len(elev_inventory),
    "elevation_measurement_rows": len(elev_measurements),
    "cleaned_wide_distinct_valid_source_keys": len(wide_set),
    "elevation_inventory_distinct_valid_source_keys": len(elev_set),
    "source_rows_only_in_elevation_inventory": len(only_elevation),
    "source_rows_only_in_cleaned_wide": len(only_cleaned),
    "source_rows_common_to_both": len(common_keys),
    "cleaned_long_records": len(clean_long),
    "elevation_measurement_records": len(elev_measurements),
    "seasonal_record_count_difference_elevation_minus_cleaned":
        len(elev_measurements) - len(clean_long),
    "source_rows_with_seasonal_count_discrepancy": len(season_discrepancies),
    "cleaned_wide_id_column": cw_id_col,
    "cleaned_wide_latitude_column": cw_lat_col,
    "cleaned_wide_longitude_column": cw_lon_col,
    "elevation_inventory_id_column": ei_id_col,
    "elevation_inventory_latitude_column": ei_lat_col,
    "elevation_inventory_longitude_column": ei_lon_col,
    "well_ids_with_coordinate_set_mismatch": int(
        (id_coordinate_comparison["coordinate_mapping_status"]
         == "COORDINATE SET MISMATCH").sum()
    ) if not id_coordinate_comparison.empty else 0,
    "well_ids_consistent_between_outputs": int(
        (id_coordinate_comparison["coordinate_mapping_status"]
         == "CONSISTENT").sum()
    ) if not id_coordinate_comparison.empty else 0,
    "well_id_coordinate_check_note":
        "Coordinates rounded to 6 decimal places; missing/invalid coordinates excluded from mapping comparison.",
}

with open(AUDIT_DIR / "reconciliation_audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

summary_table = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in summary.items()]
)
summary_table.to_csv(
    AUDIT_DIR / "reconciliation_audit_summary.csv",
    index=False
)

# ------------------------- 13. DISPLAY RESULTS ---------------------

print("\n" + "=" * 72)
print("AUDIT SUMMARY")
print("=" * 72)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSource-row count differences by workbook/sheet:")
print(file_sheet_comparison.to_string(index=False))

print("\nRows present only in elevation inventory:")
if rows_only_elevation.empty:
    print("None")
else:
    display_cols = [
        c for c in [
            "_audit_file", "_audit_sheet", "_audit_row", "_audit_source_key"
        ] if c in rows_only_elevation.columns
    ]
    print(rows_only_elevation[display_cols].to_string(index=False))

print("\nRows present only in cleaned wide output:")
if rows_only_cleaned.empty:
    print("None")
else:
    display_cols = [
        c for c in [
            "_audit_file", "_audit_sheet", "_audit_row", "_audit_source_key"
        ] if c in rows_only_cleaned.columns
    ]
    print(rows_only_cleaned[display_cols].to_string(index=False))

print("\nSeasonal-record discrepancies:")
if season_discrepancies.empty:
    print("None")
else:
    print(season_discrepancies.to_string(index=False))

print("\nWell IDs with inconsistent coordinates between outputs:")
if id_coordinate_comparison.empty:
    print("No usable well-ID/coordinate columns were identified.")
else:
    mismatch = id_coordinate_comparison[
        id_coordinate_comparison["coordinate_mapping_status"]
        != "CONSISTENT"
    ]
    print(
        mismatch.head(100).to_string(index=False)
        if not mismatch.empty else
        "No coordinate mapping mismatches detected for IDs with valid coordinates."
    )

print("\nAudit files saved in:")
print(AUDIT_DIR)
print("\nAudit completed. No source or existing output files were modified.")

## 3. Coordinate checks

**[orig #55]** Check well coordinates against the Rajasthan boundary  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
import geopandas as gpd
from pathlib import Path

# Use the combined dataset already loaded in gw
gw = gw.copy()
gw.columns = gw.columns.astype(str).str.strip()

# Convert coordinates to numeric
for c in ["Latitude", "Longitude"]:
    gw[c] = pd.to_numeric(gw[c], errors="coerce")

# Load the Rajasthan state boundary
states = gpd.read_file(
    "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IND_1.json"
)
rajasthan = states[
    states["NAME_1"].str.strip().str.lower() == "rajasthan"
]

if rajasthan.empty:
    raise ValueError("Rajasthan boundary could not be loaded.")

boundary = rajasthan.geometry.union_all()

# Check which points fall inside Rajasthan
has_coords = gw["Latitude"].notna() & gw["Longitude"].notna()
inside = pd.Series(False, index=gw.index)

pts = gpd.GeoSeries(
    gpd.points_from_xy(
        gw.loc[has_coords, "Longitude"],
        gw.loc[has_coords, "Latitude"]
    ),
    index=gw.index[has_coords],
    crs="EPSG:4326"
)

inside.loc[has_coords] = pts.apply(boundary.covers)

gw["coordinate_status"] = "inside_rajasthan"
gw.loc[~has_coords, "coordinate_status"] = "missing_coordinates"
gw.loc[has_coords & ~inside, "coordinate_status"] = "outside_rajasthan"

# Keep every record; do not delete wells
suspect = gw[gw["coordinate_status"] != "inside_rajasthan"].copy()

# Create location lookup queries from administrative names
for c in ["District", "Block", "Grampanchayat", "Village"]:
    if c not in suspect.columns:
        suspect[c] = ""

suspect["location_query"] = (
    suspect["Grampanchayat"].fillna("").astype(str).str.strip()
    + ", "
    + suspect["Block"].fillna("").astype(str).str.strip()
    + ", "
    + suspect["District"].fillna("").astype(str).str.strip()
    + ", Rajasthan, India"
)

cols = [
    c for c in [
        "WIMS_ID", "ROLTA_ID/UID", "District", "Block",
        "Grampanchayat", "Village", "Latitude", "Longitude",
        "coordinate_status", "location_query"
    ] if c in suspect.columns
]

print("Total wells:", len(gw))
print("Coordinates inside Rajasthan:", int(inside.sum()))
print("Coordinates needing review:", len(suspect))

display(suspect[cols].head(30))

suspect.to_csv(
    "/content/drive/MyDrive/gw_framework/data/"
    "Rajasthan_gw_data/coordinate_correction_review.csv",
    index=False
)
print("Review file saved.")


**[orig #56]** Suggest coordinates for suspect wells by geocoding place names (review only)  
<sub>✅ ran OK</sub>

In [ ]:

!pip -q install geopy

import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from pathlib import Path

# Start from the original combined dataset, not a dataset
# from which the suspicious records were deleted.
gw = gw.copy()

for c in ["Latitude", "Longitude"]:
    gw[c] = pd.to_numeric(gw[c], errors="coerce")

states = gpd.read_file(
    "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IND_1.json"
)
rajasthan = states[
    states["NAME_1"].str.lower() == "rajasthan"
]
boundary = rajasthan.geometry.union_all()

has_coords = gw["Latitude"].notna() & gw["Longitude"].notna()
inside = pd.Series(False, index=gw.index)

pts = gpd.GeoSeries(
    gpd.points_from_xy(
        gw.loc[has_coords, "Longitude"],
        gw.loc[has_coords, "Latitude"]
    ),
    index=gw.index[has_coords],
    crs="EPSG:4326"
)
inside.loc[has_coords] = pts.apply(boundary.covers)

review = gw.loc[~inside].copy()

geolocator = Nominatim(
    user_agent="rajasthan_groundwater_research_coordinate_review"
)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    swallow_exceptions=True
)

results = []

for idx, row in review.iterrows():
    district = str(row.get("District", "")).strip()
    block = str(row.get("Block", "")).strip()
    gp = str(row.get("Grampanchayat", "")).strip()
    village = str(row.get("Village ", row.get("Village", ""))).strip()

    # Try the most specific place name first
    queries = [
        f"{village}, {block}, {district}, Rajasthan, India",
        f"{gp}, {block}, {district}, Rajasthan, India",
        f"{block}, {district}, Rajasthan, India"
    ]

    found = None
    query_used = None

    for query in queries:
        if not query.replace(",", "").strip():
            continue

        loc = geocode(
            query,
            country_codes="in",
            addressdetails=True,
            exactly_one=True
        )

        if loc is None:
            continue

        point = gpd.points_from_xy(
            [loc.longitude], [loc.latitude]
        )[0]

        # Only suggest candidates that fall inside Rajasthan
        if boundary.covers(point):
            found = loc
            query_used = query
            break

    results.append({
        "original_index": idx,
        "WIMS_ID": row.get("WIMS_ID"),
        "District": district,
        "Block": block,
        "Grampanchayat": gp,
        "Village": village,
        "original_latitude": row.get("Latitude"),
        "original_longitude": row.get("Longitude"),
        "suggested_latitude": found.latitude if found else None,
        "suggested_longitude": found.longitude if found else None,
        "matched_place": found.address if found else None,
        "query_used": query_used,
        "correction_status": (
            "candidate_found_review_required"
            if found else "unresolved_manual_review"
        )
    })

suggestions = pd.DataFrame(results)

out = Path(
    "/content/drive/MyDrive/gw_framework/data/"
    "Rajasthan_gw_data/coordinate_suggestions.csv"
)
suggestions.to_csv(out, index=False)

display(suggestions)
print("\nSaved suggestions to:", out)


**[orig #57]** Distance check of suggested coordinates; add blank columns for manual decisions  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/gw_framework/data/Rajasthan_gw_data"
)
input_file = folder / "coordinate_suggestions.csv"
output_file = folder / "coordinate_suggestions_QC.csv"

df = pd.read_csv(input_file)

# Haversine distance in kilometres
def haversine_km(lat1, lon1, lat2, lon2):
    values = [lat1, lon1, lat2, lon2]
    if not all(pd.notna(v) for v in values):
        return np.nan

    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)

    a = np.sin(dp / 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

df["distance_km"] = df.apply(
    lambda r: haversine_km(
        r["original_latitude"],
        r["original_longitude"],
        r["suggested_latitude"],
        r["suggested_longitude"]
    ),
    axis=1
)

def classify(row):
    if pd.isna(row["suggested_latitude"]) or pd.isna(row["suggested_longitude"]):
        return "No candidate - manual review"
    if pd.isna(row["original_latitude"]) or pd.isna(row["original_longitude"]):
        return "Original coordinate missing - verify village"
    if row["distance_km"] <= 3:
        return "Nearby candidate - verify identity"
    if row["distance_km"] <= 10:
        return "Moderate shift - investigate"
    return "Large shift - likely wrong match"

df["distance_review"] = df.apply(classify, axis=1)

# Add blank fields for documented human decisions
df["verified_latitude"] = np.nan
df["verified_longitude"] = np.nan
df["verification_source"] = ""
df["review_decision"] = "Pending"

df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("\nReview categories:")
print(df["distance_review"].value_counts(dropna=False))

display(
    df[[
        "WIMS_ID", "District", "Block", "Grampanchayat", "Village",
        "original_latitude", "original_longitude",
        "suggested_latitude", "suggested_longitude",
        "distance_km", "matched_place", "distance_review"
    ]].sort_values("distance_km", na_position="last")
)


## 4. Parsed observations in `gwl/` (earlier WRIS/CGWB route)

**[orig #39]** Load `gwl/diagnostics/manual_observations_audit.csv`  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd

file_path = (
    "/content/drive/MyDrive/gw_framework/data/"
    "gwl/diagnostics/manual_observations_audit.csv"
)

df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

**[orig #40]** DTW quality audit (distribution, negative DTW by well) → `gwl/diagnostics`  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
from pathlib import Path

file_path = (
    "/content/drive/MyDrive/gw_framework/data/"
    "gwl/diagnostics/manual_observations_audit.csv"
)

df = pd.read_csv(file_path)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["dtw"] = pd.to_numeric(df["dtw"], errors="coerce")

print("=" * 60)
print("GROUNDWATER OBSERVATION QUALITY AUDIT")
print("=" * 60)

# 1. Basic summary
print("\n1. DATA SUMMARY")
print("Total observations:", len(df))
print("Number of wells:", df["well_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Missing DTW:", df["dtw"].isna().sum())
print("Duplicate well-date records:",
      df.duplicated(["well_id", "date"]).sum())

# 2. DTW distribution
print("\n2. DTW DISTRIBUTION (metres)")
print(df["dtw"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nNegative DTW observations:", (df["dtw"] < 0).sum())
print("Zero DTW observations:", (df["dtw"] == 0).sum())
print("DTW above 50 m:", (df["dtw"] > 50).sum())
print("DTW above 100 m:", (df["dtw"] > 100).sum())

# 3. Negative DTW values by well
print("\n3. NEGATIVE DTW BY WELL")
negative = df[df["dtw"] < 0]

if len(negative):
    display(
        negative.groupby("well_id")
        .agg(
            negative_count=("dtw", "size"),
            minimum_dtw=("dtw", "min"),
            first_date=("date", "min"),
            last_date=("date", "max")
        )
        .sort_values("negative_count", ascending=False)
    )
else:
    print("No negative DTW values found.")

# 4. Extreme positive values
print("\n4. 15 LARGEST DTW READINGS")
display(
    df.nlargest(15, "dtw")[["well_id", "date", "dtw"]]
)

# 5. Dry-well flags
print("\n5. DRY-WELL FLAG COUNTS")
print(df["dry"].value_counts(dropna=False))

# 6. Save results
output_dir = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics"
)
output_dir.mkdir(parents=True, exist_ok=True)

negative.to_csv(
    output_dir / "negative_dtw_records.csv",
    index=False
)

print("\nNegative DTW records saved to:")
print(output_dir / "negative_dtw_records.csv")

**[orig #41]** Per-well summary of parsed observations (`gwl/processed/_parsed`)  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/gw_framework/data")
PARSED = ROOT / "gwl" / "processed" / "_parsed"

# Load the original parsed groundwater observations
obs_files = list(PARSED.glob("*_man.parquet"))
meta_files = list(PARSED.glob("*_meta.parquet"))

if not obs_files or not meta_files:
    raise FileNotFoundError(
        f"Could not find observation and metadata Parquet files in {PARSED}"
    )

obs = pd.read_parquet(obs_files[0])
meta = pd.read_parquet(meta_files[0])

obs["date"] = pd.to_datetime(obs["date"], errors="coerce")
obs["dtw"] = pd.to_numeric(obs["dtw"], errors="coerce")

# Summarize each well
summary = (
    obs.groupby("well_id")
    .agg(
        n_observations=("dtw", "count"),
        negative_count=("dtw", lambda x: (x < 0).sum()),
        minimum_dtw=("dtw", "min"),
        median_dtw=("dtw", "median"),
        maximum_dtw=("dtw", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

# Add available well metadata
metadata_columns = [
    c for c in [
        "well_id", "name", "lat", "lon", "state", "district",
        "block", "well_type", "aquifer", "well_depth_m", "elev_user"
    ]
    if c in meta.columns
]

report = summary.merge(
    meta[metadata_columns].drop_duplicates("well_id"),
    on="well_id",
    how="left",
    indicator=True
)

print("WELLS WITH MOST NEGATIVE READINGS")
display(
    report.sort_values("negative_count", ascending=False)
)

print("\nWELLS WITH DTW EXCEEDING RECORDED WELL DEPTH")
if "well_depth_m" in report.columns:
    depth = pd.to_numeric(report["well_depth_m"], errors="coerce")
    flagged = report[
        report["maximum_dtw"].gt(depth) & depth.notna()
    ]
    display(
        flagged[
            ["well_id", "minimum_dtw", "maximum_dtw", "well_depth_m"]
        ]
    )
else:
    print("Well depth metadata is unavailable.")

print("\nCOORDINATE EXTENT")
if {"lat", "lon"}.issubset(meta.columns):
    print("Latitude:", meta["lat"].min(), "to", meta["lat"].max())
    print("Longitude:", meta["lon"].min(), "to", meta["lon"].max())

# Save a report for further review
out = ROOT / "gwl" / "diagnostics"
out.mkdir(parents=True, exist_ok=True)
report.to_csv(out / "well_metadata_validation.csv", index=False)

print("\nSaved:", out / "well_metadata_validation.csv")

**[orig #73]** Study-area and well-metadata verification (read-only)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# GROUNDWATER STUDY-AREA AND WELL-METADATA VERIFICATION
# Google Colab | Read-only source audit
#
# Checks:
# 1. Find metadata and groundwater CSV files
# 2. Validate latitude / longitude
# 3. Inspect district, state, block and source labels
# 4. Identify duplicate IDs and conflicting coordinates
# 5. Compare locations with a broad Alwar screening box
# 6. Save reports to Google Drive
#
# No source files are modified and no coordinates are changed.
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# ------------------------ PATHS ----------------------------------

ROOT = Path("/content/drive/MyDrive/gw_framework/data/gwl")

OUT = ROOT / "diagnostics" / "Study_Area_Verification"
OUT.mkdir(parents=True, exist_ok=True)

if not ROOT.exists():
    raise FileNotFoundError(
        f"Groundwater folder not found: {ROOT}\n"
        "Check the folder path in Google Drive."
    )

print("Source folder:", ROOT)
print("Report folder:", OUT)

# ------------------------ 1. INVENTORY ---------------------------

csv_files = sorted(ROOT.rglob("*.csv"))

# Exclude outputs from this verification run to avoid rereading reports.
csv_files = [
    p for p in csv_files
    if OUT not in p.parents
]

inventory = []

for p in csv_files:
    try:
        sample = pd.read_csv(p, nrows=5, low_memory=False)
        inventory.append({
            "file": str(p),
            "filename": p.name,
            "rows_in_sample": len(sample),
            "columns": len(sample.columns),
            "column_names": " | ".join(map(str, sample.columns)),
            "read_status": "OK"
        })
    except Exception as e:
        inventory.append({
            "file": str(p),
            "filename": p.name,
            "rows_in_sample": None,
            "columns": None,
            "column_names": "",
            "read_status": f"ERROR: {e}"
        })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(OUT / "01_csv_inventory.csv", index=False)

print("\nCSV files found:", len(csv_files))
for p in csv_files:
    print(" -", p.relative_to(ROOT))

# ------------------------ 2. COLUMN MATCHING ---------------------

def normalize_column(name):
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

def find_column(columns, aliases):
    normalized = {c: normalize_column(c) for c in columns}

    # Prefer exact matches.
    for alias in aliases:
        for original, norm in normalized.items():
            if norm == alias:
                return original

    # Then accept common prefixed/suffixed variants.
    for alias in aliases:
        for original, norm in normalized.items():
            if alias in norm:
                return original

    return None

lat_aliases = ["latitude", "lat"]
lon_aliases = ["longitude", "lon", "long"]
id_aliases = ["well_id", "wellid", "well_code", "site_id", "station_id"]
district_aliases = ["district", "district_name"]
state_aliases = ["state", "state_name"]
block_aliases = ["block", "block_name", "tehsil", "taluka"]
name_aliases = ["name", "well_name", "station_name"]
source_aliases = ["source", "agency", "file"]

# ------------------------ 3. IDENTIFY METADATA -------------------

metadata_candidates = []

for p in csv_files:
    try:
        head = pd.read_csv(p, nrows=5, low_memory=False)
        cols = list(head.columns)

        lat = find_column(cols, lat_aliases)
        lon = find_column(cols, lon_aliases)
        wid = find_column(cols, id_aliases)
        district = find_column(cols, district_aliases)
        state = find_column(cols, state_aliases)

        score = sum([
            lat is not None,
            lon is not None,
            wid is not None,
            district is not None,
            state is not None
        ])

        metadata_candidates.append({
            "path": p,
            "score": score,
            "latitude_column": lat,
            "longitude_column": lon,
            "well_id_column": wid,
            "district_column": district,
            "state_column": state,
            "columns": cols
        })
    except Exception:
        pass

metadata_candidates.sort(
    key=lambda x: (
        x["score"],
        "metadata" in x["path"].name.lower(),
        "well" in x["path"].name.lower()
    ),
    reverse=True
)

if not metadata_candidates:
    raise RuntimeError("No readable CSV files were found.")

print("\nPotential metadata files:")
for item in metadata_candidates[:10]:
    print(
        item["path"].name,
        "| score:", item["score"],
        "| lat:", item["latitude_column"],
        "| lon:", item["longitude_column"],
        "| ID:", item["well_id_column"]
    )

# Select the strongest candidate containing both coordinates.
usable = [
    x for x in metadata_candidates
    if x["latitude_column"] and x["longitude_column"]
]

if not usable:
    raise RuntimeError(
        "No CSV with recognizable latitude and longitude columns was found. "
        "Review 01_csv_inventory.csv and specify the correct file manually."
    )

meta_info = usable[0]
META_PATH = meta_info["path"]

print("\nSelected metadata file:", META_PATH)
print("Detected columns:", meta_info["columns"])

wells = pd.read_csv(META_PATH, low_memory=False)

lat_col = meta_info["latitude_column"]
lon_col = meta_info["longitude_column"]
id_col = meta_info["well_id_column"]

district_col = find_column(wells.columns, district_aliases)
state_col = find_column(wells.columns, state_aliases)
block_col = find_column(wells.columns, block_aliases)
name_col = find_column(wells.columns, name_aliases)
source_col = find_column(wells.columns, source_aliases)

# ------------------------ 4. STANDARDIZE FOR AUDIT ---------------

audit = wells.copy()

audit["audit_latitude"] = pd.to_numeric(
    audit[lat_col], errors="coerce"
)
audit["audit_longitude"] = pd.to_numeric(
    audit[lon_col], errors="coerce"
)

if id_col:
    audit["audit_well_id"] = audit[id_col].astype("string").str.strip()
else:
    audit["audit_well_id"] = pd.NA

# Broad geographic validity checks, not district membership checks.
audit["missing_coordinate"] = (
    audit["audit_latitude"].isna()
    | audit["audit_longitude"].isna()
)

audit["latitude_out_of_range"] = (
    audit["audit_latitude"].notna()
    & ~audit["audit_latitude"].between(-90, 90)
)

audit["longitude_out_of_range"] = (
    audit["audit_longitude"].notna()
    & ~audit["audit_longitude"].between(-180, 180)
)

# Approximate screening rectangle only. It is NOT an official Alwar boundary.
audit["inside_approx_alwar_screen"] = (
    audit["audit_latitude"].between(27.0, 28.1)
    & audit["audit_longitude"].between(76.0, 77.2)
)

audit["outside_approx_alwar_screen"] = (
    ~audit["inside_approx_alwar_screen"]
    & ~audit["missing_coordinate"]
    & ~audit["latitude_out_of_range"]
    & ~audit["longitude_out_of_range"]
)

if district_col:
    audit["audit_district"] = audit[district_col].astype("string").str.strip()
else:
    audit["audit_district"] = pd.NA

if state_col:
    audit["audit_state"] = audit[state_col].astype("string").str.strip()
else:
    audit["audit_state"] = pd.NA

if block_col:
    audit["audit_block"] = audit[block_col].astype("string").str.strip()
else:
    audit["audit_block"] = pd.NA

# ------------------------ 5. DUPLICATE / CONFLICT CHECKS ---------

if id_col:
    nonblank_id = (
        audit["audit_well_id"].notna()
        & audit["audit_well_id"].ne("")
    )

    audit["duplicate_well_id"] = (
        nonblank_id
        & audit["audit_well_id"].duplicated(keep=False)
    )

    coordinate_spread = (
        audit.loc[nonblank_id]
        .groupby("audit_well_id")
        .agg(
            records=("audit_well_id", "size"),
            unique_latitudes=("audit_latitude", "nunique"),
            unique_longitudes=("audit_longitude", "nunique"),
            min_latitude=("audit_latitude", "min"),
            max_latitude=("audit_latitude", "max"),
            min_longitude=("audit_longitude", "min"),
            max_longitude=("audit_longitude", "max")
        )
        .reset_index()
    )

    coordinate_spread["conflicting_coordinates"] = (
        (coordinate_spread["unique_latitudes"] > 1)
        | (coordinate_spread["unique_longitudes"] > 1)
    )
else:
    audit["duplicate_well_id"] = False
    coordinate_spread = pd.DataFrame()

audit.to_csv(
    OUT / "02_well_location_audit.csv",
    index=False
)

coordinate_spread.to_csv(
    OUT / "03_duplicate_id_coordinate_check.csv",
    index=False
)

# ------------------------ 6. LABEL SUMMARIES ---------------------

def frequency_table(series, label):
    if series is None:
        return pd.DataFrame(columns=[label, "count"])

    return (
        series.fillna("[MISSING]")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(label)
        .reset_index(name="count")
    )

district_summary = frequency_table(
    audit["audit_district"] if district_col else None,
    "district"
)
state_summary = frequency_table(
    audit["audit_state"] if state_col else None,
    "state"
)
block_summary = frequency_table(
    audit["audit_block"] if block_col else None,
    "block"
)

district_summary.to_csv(OUT / "04_district_labels.csv", index=False)
state_summary.to_csv(OUT / "05_state_labels.csv", index=False)
block_summary.to_csv(OUT / "06_block_labels.csv", index=False)

# ------------------------ 7. LOCATION SUMMARY --------------------

valid = audit.loc[
    ~audit["missing_coordinate"]
    & ~audit["latitude_out_of_range"]
    & ~audit["longitude_out_of_range"]
]

summary = {
    "metadata_file": str(META_PATH),
    "metadata_rows": int(len(audit)),
    "unique_nonblank_well_ids": (
        int(audit.loc[
            audit["audit_well_id"].notna()
            & audit["audit_well_id"].ne(""),
            "audit_well_id"
        ].nunique())
        if id_col else None
    ),
    "missing_coordinates": int(audit["missing_coordinate"].sum()),
    "invalid_latitude_rows": int(audit["latitude_out_of_range"].sum()),
    "invalid_longitude_rows": int(audit["longitude_out_of_range"].sum()),
    "duplicate_id_rows": int(audit["duplicate_well_id"].sum()),
    "valid_coordinate_rows": int(len(valid)),
    "latitude_min": (
        float(valid["audit_latitude"].min()) if len(valid) else None
    ),
    "latitude_max": (
        float(valid["audit_latitude"].max()) if len(valid) else None
    ),
    "longitude_min": (
        float(valid["audit_longitude"].min()) if len(valid) else None
    ),
    "longitude_max": (
        float(valid["audit_longitude"].max()) if len(valid) else None
    ),
    "inside_approx_alwar_screen": int(
        audit["inside_approx_alwar_screen"].sum()
    ),
    "outside_approx_alwar_screen": int(
        audit["outside_approx_alwar_screen"].sum()
    ),
    "district_column_detected": district_col,
    "state_column_detected": state_col,
    "block_column_detected": block_col,
    "screening_note": (
        "Approximate coordinate rectangle only; not an official district boundary."
    )
}

with open(OUT / "07_study_area_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ------------------------ 8. PRINT RESULTS -----------------------

print("\n" + "=" * 65)
print("STUDY-AREA VERIFICATION SUMMARY")
print("=" * 65)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nDISTRICT LABELS")
print(district_summary.to_string(index=False))

print("\nSTATE LABELS")
print(state_summary.to_string(index=False))

print("\nBLOCK LABELS")
print(block_summary.to_string(index=False))

print("\nWELL LOCATIONS")
display_cols = [
    c for c in [
        "audit_well_id",
        "audit_latitude",
        "audit_longitude",
        "audit_state",
        "audit_district",
        "audit_block",
        "inside_approx_alwar_screen",
        "duplicate_well_id"
    ]
    if c in audit.columns
]

print(audit[display_cols].to_string(index=False))

print("\nReports saved to:")
print(OUT)

print("\nImportant:")
print(
    "The screening rectangle is not a district polygon. "
    "Confirm locations against an official boundary before calibration."
)


**[orig #75]** Groundwater source recovery audit: find non-empty sources, diagnose empty processed files  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# GROUNDWATER SOURCE RECOVERY AUDIT
# Search for non-empty source files and diagnose empty processed files
# ================================================================

from pathlib import Path
import pandas as pd
import os

BASE = Path("/content/drive/MyDrive/gw_framework")
OUT = BASE / "data/gwl/diagnostics/Study_Area_Verification"
OUT.mkdir(parents=True, exist_ok=True)

print("Searching:", BASE)

# Search the entire project folder, not just processed/gwl.
extensions = {".csv", ".xlsx", ".xls", ".parquet", ".json"}

records = []

for p in BASE.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in extensions:
        continue

    # Skip generated verification reports to avoid recursing through outputs.
    if OUT in p.parents:
        continue

    try:
        size = p.stat().st_size
    except OSError:
        continue

    row = {
        "path": str(p),
        "filename": p.name,
        "extension": p.suffix.lower(),
        "size_bytes": size,
        "rows": None,
        "columns": None,
        "column_names": "",
        "read_status": ""
    }

    if size == 0:
        row["read_status"] = "ZERO_BYTE_FILE"
        records.append(row)
        continue

    try:
        if p.suffix.lower() == ".csv":
            # Read the header first, then count rows without loading all data.
            header = pd.read_csv(p, nrows=0, low_memory=False)
            row["columns"] = len(header.columns)
            row["column_names"] = " | ".join(map(str, header.columns))

            with open(p, "rb") as f:
                row["rows"] = max(
                    sum(1 for _ in f) - 1, 0
                )
            row["read_status"] = (
                "NONEMPTY_DATA" if row["rows"] > 0 else "HEADER_ONLY"
            )

        elif p.suffix.lower() in {".xlsx", ".xls"}:
            xls = pd.ExcelFile(p)
            details = []
            total_rows = 0

            for sheet in xls.sheet_names:
                try:
                    df = pd.read_excel(p, sheet_name=sheet)
                    details.append(
                        f"{sheet}: {len(df)} rows x {len(df.columns)} columns"
                    )
                    total_rows += len(df)
                except Exception as e:
                    details.append(f"{sheet}: ERROR {e}")

            row["rows"] = total_rows
            row["column_names"] = " ; ".join(details)
            row["read_status"] = (
                "NONEMPTY_DATA" if total_rows > 0 else "NO_DATA_ROWS"
            )

        elif p.suffix.lower() == ".parquet":
            df = pd.read_parquet(p)
            row["rows"] = len(df)
            row["columns"] = len(df.columns)
            row["column_names"] = " | ".join(map(str, df.columns))
            row["read_status"] = (
                "NONEMPTY_DATA" if len(df) > 0 else "NO_DATA_ROWS"
            )

        else:
            row["read_status"] = "JSON_FILE_NOT_PARSED"

    except Exception as e:
        row["read_status"] = f"READ_ERROR: {str(e)[:250]}"

    records.append(row)

inventory = pd.DataFrame(records)

if inventory.empty:
    print("No matching files found.")
else:
    inventory = inventory.sort_values(
        ["read_status", "size_bytes"],
        ascending=[True, False]
    )

    inventory.to_csv(
        OUT / "09_project_source_file_inventory.csv",
        index=False
    )

    print("\nFile count:", len(inventory))
    print("\nStatus counts:")
    print(inventory["read_status"].value_counts(dropna=False))

    print("\nLargest non-empty data files:")
    display(
        inventory[inventory["size_bytes"] > 0]
        .sort_values("size_bytes", ascending=False)
        .head(40)
    )

    print("\nLikely groundwater files:")
    keywords = (
        "groundwater", "water_level", "waterlevel", "well",
        "observation", "seasonal", "dtw", "gwl", "manual",
        "cgwb", "dwlr", "piezometer"
    )

    mask = inventory["path"].str.lower().apply(
        lambda x: any(k in x for k in keywords)
    )

    display(
        inventory.loc[mask, [
            "path", "size_bytes", "rows",
            "columns", "read_status"
        ]].head(100)
    )

# ------------------------------------------------
# Corrected comparison of the two metadata files
# ------------------------------------------------

original_path = BASE / "data/gwl/processed/wells_meta.csv"
validation_path = (
    BASE / "data/gwl/diagnostics/well_metadata_validation.csv"
)

if original_path.exists() and validation_path.exists():
    original = pd.read_csv(original_path, low_memory=False)
    validation = pd.read_csv(validation_path, low_memory=False)

    print("\nOriginal metadata shape:", original.shape)
    print("Validation metadata shape:", validation.shape)

    if "well_id" in original.columns and "well_id" in validation.columns:
        # Remove any pre-existing merge indicator to avoid the earlier error.
        left = original.drop(columns=["_merge"], errors="ignore").copy()
        right = validation.drop(columns=["_merge"], errors="ignore").copy()

        left["well_id"] = left["well_id"].astype("string").str.strip()
        right["well_id"] = right["well_id"].astype("string").str.strip()

        comparison = left.merge(
            right,
            on="well_id",
            how="outer",
            suffixes=("_original", "_validation"),
            indicator="metadata_merge_status"
        )

        print("\nMetadata match status:")
        print(comparison["metadata_merge_status"].value_counts())

        # Compare coordinates only if both files have coordinate columns.
        for col in ["lat", "lon"]:
            a, b = f"{col}_original", f"{col}_validation"
            if a in comparison.columns and b in comparison.columns:
                comparison[f"{col}_difference"] = (
                    pd.to_numeric(comparison[a], errors="coerce")
                    - pd.to_numeric(comparison[b], errors="coerce")
                )

        comparison.to_csv(
            OUT / "10_original_vs_validation_metadata.csv",
            index=False
        )

        print("\nSaved metadata comparison.")
        display(comparison.head(30))

print("\nAudit completed.")
print("Reports folder:", OUT)


**[orig #79]** Check the expected `gwl` files exist  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive/gw_framework/data/gwl")
out = root / "diagnostics" / "Study_Area_Verification"
out.mkdir(parents=True, exist_ok=True)

files = {
    "seasonal_csv": root / "processed" / "seasonal.csv",
    "seasonal_parquet": root / "processed" / "seasonal.parquet",
    "manual_readings_parquet": root / "processed" / "readings_manual_qc.parquet",
    "dwlr_readings_parquet": root / "processed" / "dwlr_daily_qc.parquet",
    "manual_observations_audit": root / "diagnostics" / "manual_observations_audit.csv",
}

report = []

for name, path in files.items():
    print("\n" + "=" * 75)
    print(f"{name}: {path}")
    print("=" * 75)

    if not path.exists():
        print("FILE MISSING")
        report.append({
            "dataset": name, "path": str(path),
            "rows": None, "columns": None, "status": "MISSING"
        })
        continue

    try:
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path, low_memory=False)

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("\nFirst 5 records:")
        display(df.head(5))

        report.append({
            "dataset": name,
            "path": str(path),
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": " | ".join(map(str, df.columns)),
            "status": "NONEMPTY" if len(df) else "EMPTY"
        })

    except Exception as e:
        print("READ ERROR:", repr(e))
        report.append({
            "dataset": name, "path": str(path),
            "rows": None, "columns": None,
            "status": f"ERROR: {e}"
        })

report_df = pd.DataFrame(report)
report_path = out / "11_groundwater_source_inspection.csv"
report_df.to_csv(report_path, index=False)

print("\n" + "=" * 75)
print("INSPECTION SUMMARY")
print("=" * 75)
display(report_df)

print("Report saved to:", report_path)
print("\nNo source files were modified.")


**[orig #82]** Search likely source folders → `12_likely_groundwater_source_files.csv`  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/gw_framework")
MYDRIVE = Path("/content/drive/MyDrive")

print("Searching for groundwater source files...")

# Search likely source locations
search_roots = [
    BASE / "data" / "gwl" / "raw_wris",
    BASE / "data" / "gwl" / "processed" / "_parsed",
    BASE / "data" / "inputs",
    BASE / "data" / "gwl"
]

extensions = {".csv", ".xlsx", ".xls", ".parquet", ".zip"}
results = []
seen = set()

for root in search_roots:
    if not root.exists():
        print("Folder not found:", root)
        continue

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in extensions:
            continue

        resolved = str(p.resolve())
        if resolved in seen:
            continue
        seen.add(resolved)

        try:
            size = p.stat().st_size
        except OSError:
            size = -1

        results.append({
            "filename": p.name,
            "path": str(p),
            "size_bytes": size,
            "extension": p.suffix.lower()
        })

df = pd.DataFrame(results)

print("\nFILES FOUND:", len(df))

if not df.empty:
    df = df.sort_values("size_bytes", ascending=False)
    display(df.head(100))

    report = (
        BASE / "data" / "gwl" / "diagnostics"
        / "Study_Area_Verification"
    )
    report.mkdir(parents=True, exist_ok=True)
    df.to_csv(report / "12_likely_groundwater_source_files.csv",
              index=False)
else:
    print("No matching files found in the searched folders.")

# Search for the exact original manual-observation filename.
print("\nSEARCHING FOR ORIGINAL OBSERVATION FILE")

matches = [
    p for p in MYDRIVE.rglob("wells_levels_user_original.csv")
    if p.is_file()
]

if matches:
    for p in matches:
        print("\nFOUND:", p)
        source = pd.read_csv(p, low_memory=False)
        print("Shape:", source.shape)
        print("Columns:", source.columns.tolist())
        display(source.head(5))
else:
    print("Exact filename not found in My Drive.")

print("\nSearch complete. Existing source files were not modified.")


**[orig #83]** Summarise the recovered source file by well  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ------------------------------------------------
# Paths
# ------------------------------------------------
SOURCE = Path(
    "/content/drive/MyDrive/gw_framework/inputs/gwl/"
    "wells_levels_user_original.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Study_Area_Verification/Recovery_Precheck"
)
OUT.mkdir(parents=True, exist_ok=True)

if not SOURCE.exists():
    raise FileNotFoundError(f"Original data not found: {SOURCE}")

# ------------------------------------------------
# Read source without modifying it
# ------------------------------------------------
df = pd.read_csv(SOURCE, low_memory=False)

required = [
    "well_id", "lat", "lon", "date",
    "dtw_mbgl", "well_depth_m"
]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Required source columns missing: {missing}")

df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
df["lat_numeric"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon_numeric"] = pd.to_numeric(df["lon"], errors="coerce")
df["dtw_numeric"] = pd.to_numeric(df["dtw_mbgl"], errors="coerce")
df["well_depth_numeric"] = pd.to_numeric(
    df["well_depth_m"], errors="coerce"
)

# ------------------------------------------------
# Source QC checks
# ------------------------------------------------
summary = {
    "source_file": str(SOURCE),
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "unique_wells": int(df["well_id"].nunique(dropna=True)),
    "duplicate_well_date_pairs": int(
        df.duplicated(["well_id", "date"]).sum()
    ),
    "invalid_dates": int(df["date_parsed"].isna().sum()),
    "invalid_latitudes": int(
        (~df["lat_numeric"].between(-90, 90)).sum()
        + df["lat_numeric"].isna().sum()
    ),
    "invalid_longitudes": int(
        (~df["lon_numeric"].between(-180, 180)).sum()
        + df["lon_numeric"].isna().sum()
    ),
    "missing_dtw": int(df["dtw_numeric"].isna().sum()),
    "negative_dtw": int((df["dtw_numeric"] < 0).sum()),
    "zero_dtw": int((df["dtw_numeric"] == 0).sum()),
    "minimum_dtw": (
        float(df["dtw_numeric"].min())
        if df["dtw_numeric"].notna().any() else None
    ),
    "maximum_dtw": (
        float(df["dtw_numeric"].max())
        if df["dtw_numeric"].notna().any() else None
    ),
    "latitude_min": (
        float(df["lat_numeric"].min())
        if df["lat_numeric"].notna().any() else None
    ),
    "latitude_max": (
        float(df["lat_numeric"].max())
        if df["lat_numeric"].notna().any() else None
    ),
    "longitude_min": (
        float(df["lon_numeric"].min())
        if df["lon_numeric"].notna().any() else None
    ),
    "longitude_max": (
        float(df["lon_numeric"].max())
        if df["lon_numeric"].notna().any() else None
    ),
    "first_date": (
        str(df["date_parsed"].min().date())
        if df["date_parsed"].notna().any() else None
    ),
    "last_date": (
        str(df["date_parsed"].max().date())
        if df["date_parsed"].notna().any() else None
    )
}

# Approximate screening only; not an official boundary.
df["approx_alwar_screen"] = (
    df["lat_numeric"].between(27.0, 28.1)
    & df["lon_numeric"].between(76.0, 77.2)
)

summary["observations_inside_approx_alwar_screen"] = int(
    df["approx_alwar_screen"].sum()
)
summary["observations_outside_approx_alwar_screen"] = int(
    (~df["approx_alwar_screen"]).sum()
)

with open(OUT / "recovery_precheck_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save per-well coordinate summary
well_summary = (
    df.groupby("well_id", dropna=False)
    .agg(
        observations=("date", "size"),
        lat_min=("lat_numeric", "min"),
        lat_max=("lat_numeric", "max"),
        lon_min=("lon_numeric", "min"),
        lon_max=("lon_numeric", "max"),
        first_date=("date_parsed", "min"),
        last_date=("date_parsed", "max"),
        negative_dtw_count=("dtw_numeric", lambda x: int((x < 0).sum())),
        well_depth_m=("well_depth_numeric", "median")
    )
    .reset_index()
)

well_summary.to_csv(
    OUT / "recovery_precheck_by_well.csv", index=False
)

print("SOURCE SUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nPER-WELL SUMMARY")
display(well_summary)

print("\nReports saved to:", OUT)
print("Original file remains unchanged.")


## 5. Alwar source recovery — Steps 2 to 6

**[orig #85]** **MAIN** — Alwar source recovery and geographic screening  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# ALWAR GROUNDWATER SOURCE RECOVERY AND GEOGRAPHIC SCREENING
# Google Colab | Read-only source audit
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/gw_framework")

OUT = ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery"
OUT.mkdir(parents=True, exist_ok=True)

# Provisional screening limits only; not the official district boundary.
LAT_MIN, LAT_MAX = 27.0, 28.1
LON_MIN, LON_MAX = 76.0, 77.2

EXCLUDED_FILENAME = "wells_levels_user_original.csv"

SUPPORTED = {".csv", ".xlsx", ".xls", ".parquet"}
inventory = []
errors = []
candidate_frames = []

def normalized_name(value):
    return "".join(ch.lower() for ch in str(value) if ch.isalnum())

def find_column(columns, aliases):
    lookup = {normalized_name(c): c for c in columns}
    for alias in aliases:
        if normalized_name(alias) in lookup:
            return lookup[normalized_name(alias)]
    return None

LAT_ALIASES = ["lat", "latitude", "latitude_dd", "well_lat", "lat_dd"]
LON_ALIASES = ["lon", "long", "longitude", "longitude_dd",
                "well_lon", "lon_dd"]
ID_ALIASES = ["well_id", "wellid", "station_id", "site_id",
               "observation_well_id", "id"]
DATE_ALIASES = ["date", "observation_date", "measurement_date",
                "sample_date", "datetime"]
DTW_ALIASES = ["dtw", "dtw_mbgl", "depth_to_water",
               "depth_to_water_level", "water_level_depth",
               "waterlevel", "water_level", "wl", "mbgl",
               "depth_bgl", "groundwater_level"]

def inspect_dataframe(df, path, sheet=""):
    cols = list(df.columns)
    lat_col = find_column(cols, LAT_ALIASES)
    lon_col = find_column(cols, LON_ALIASES)
    id_col = find_column(cols, ID_ALIASES)
    date_col = find_column(cols, DATE_ALIASES)
    dtw_col = find_column(cols, DTW_ALIASES)

    has_gw_fields = dtw_col is not None
    has_location = lat_col is not None and lon_col is not None

    row = {
        "file": str(path),
        "sheet": sheet,
        "rows": len(df),
        "columns": len(cols),
        "column_names": " | ".join(map(str, cols)),
        "well_id_column": id_col,
        "date_column": date_col,
        "dtw_column": dtw_col,
        "latitude_column": lat_col,
        "longitude_column": lon_col,
        "has_groundwater_level_field": has_gw_fields,
        "has_coordinate_fields": has_location,
        "excluded_source_file": path.name.lower() == EXCLUDED_FILENAME.lower(),
        "lat_min": np.nan,
        "lat_max": np.nan,
        "lon_min": np.nan,
        "lon_max": np.nan,
        "rows_inside_provisional_bbox": np.nan,
        "distinct_wells": np.nan,
        "first_date": None,
        "last_date": None,
    }

    if has_location:
        lat = pd.to_numeric(df[lat_col], errors="coerce")
        lon = pd.to_numeric(df[lon_col], errors="coerce")
        valid = lat.between(-90, 90) & lon.between(-180, 180)

        if valid.any():
            row["lat_min"] = float(lat[valid].min())
            row["lat_max"] = float(lat[valid].max())
            row["lon_min"] = float(lon[valid].min())
            row["lon_max"] = float(lon[valid].max())

        inside = (
            lat.between(LAT_MIN, LAT_MAX)
            & lon.between(LON_MIN, LON_MAX)
        )
        row["rows_inside_provisional_bbox"] = int(inside.sum())

        # Keep a separate copy of potential groundwater observations.
        if has_gw_fields and not row["excluded_source_file"]:
            subset = df.loc[inside].copy()
            if len(subset):
                subset["_audit_source_file"] = str(path)
                subset["_audit_sheet"] = sheet
                candidate_frames.append(subset)

    if id_col:
        row["distinct_wells"] = int(df[id_col].nunique(dropna=True))

    if date_col:
        dates = pd.to_datetime(df[date_col], errors="coerce")
        if dates.notna().any():
            row["first_date"] = str(dates.min().date())
            row["last_date"] = str(dates.max().date())

    return row

# 2. Search project files without touching originals
files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in SUPPORTED
    and OUT not in p.parents
)

print(f"Project root: {ROOT}")
print(f"Supported files found: {len(files)}")

# 3. Inspect each file and each Excel sheet
for i, path in enumerate(files, 1):
    try:
        print(f"[{i}/{len(files)}] {path.relative_to(ROOT)}")

        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, low_memory=False)
            inventory.append(inspect_dataframe(df, path))

        elif path.suffix.lower() in {".xlsx", ".xls"}:
            sheets = pd.read_excel(path, sheet_name=None)
            for sheet, df in sheets.items():
                inventory.append(inspect_dataframe(df, path, sheet))

        elif path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
            inventory.append(inspect_dataframe(df, path))

    except Exception as e:
        errors.append({
            "file": str(path),
            "error": repr(e)
        })

# 4. Save inventory and errors
inv = pd.DataFrame(inventory)
inv_path = OUT / "01_groundwater_source_inventory.csv"
inv.to_csv(inv_path, index=False)

err_path = OUT / "02_source_inspection_errors.csv"
pd.DataFrame(errors).to_csv(err_path, index=False)

# 5. Create shortlist of files with groundwater-level fields
if len(inv):
    shortlist = inv[
        inv["has_groundwater_level_field"] == True
    ].copy()

    shortlist = shortlist.sort_values(
        ["has_coordinate_fields", "rows_inside_provisional_bbox",
         "rows"],
        ascending=[False, False, False],
        na_position="last"
    )
else:
    shortlist = pd.DataFrame()

shortlist_path = OUT / "03_groundwater_source_shortlist.csv"
shortlist.to_csv(shortlist_path, index=False)

# 6. Combine provisional in-area observations from candidate sources
if candidate_frames:
    combined = pd.concat(candidate_frames, ignore_index=True, sort=False)
    combined_path = OUT / "04_provisional_in_bbox_candidates.csv"
    combined.to_csv(combined_path, index=False)
else:
    combined = pd.DataFrame()
    combined_path = OUT / "04_provisional_in_bbox_candidates.csv"
    combined.to_csv(combined_path, index=False)

# 7. Summary report
summary = {
    "project_root": str(ROOT),
    "supported_files_found": len(files),
    "files_inspected": len(inv),
    "inspection_errors": len(errors),
    "groundwater_level_file_sheets": len(shortlist),
    "candidate_rows_inside_provisional_bbox": len(combined),
    "provisional_bbox": {
        "latitude_min": LAT_MIN,
        "latitude_max": LAT_MAX,
        "longitude_min": LON_MIN,
        "longitude_max": LON_MAX
    },
    "excluded_filename": EXCLUDED_FILENAME,
    "important_note": (
        "Bounding-box matches are candidates only. Verify coordinates "
        "against the official Alwar district boundary before analysis."
    ),
    "outputs": [
        str(inv_path), str(err_path),
        str(shortlist_path), str(combined_path)
    ]
}

with open(OUT / "05_recovery_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("SOURCE RECOVERY AUDIT COMPLETE")
print("=" * 65)
print(json.dumps(summary, indent=2))

if len(shortlist):
    display(shortlist[
        ["file", "sheet", "rows", "column_names",
         "has_coordinate_fields", "rows_inside_provisional_bbox",
         "first_date", "last_date", "excluded_source_file"]
    ].head(50))
else:
    print("No files with recognized groundwater-level columns were found.")

print("\nResults saved to:")
print(OUT)


**[orig #86]** Open candidate source workbooks and list sheets  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive/gw_framework")

targets = [
    root / "data/Rajasthan_gw_data/Alwar.xlsx",
    root / "data/BDR/Alwar/nhpbdr 2023/Alwar NHP.xlsx",
    root / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/groundwater_measurements_with_estimated_head.csv",
    root / "data/Rajasthan_gw_data/Groundwater_Cleaning_According_to_User_Rules/groundwater_cleaned_long_seasonal.csv",
]

for path in targets:
    print("\n" + "=" * 80)
    print("FILE:", path)
    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        if path.suffix.lower() == ".xlsx":
            sheets = pd.read_excel(path, sheet_name=None)
            for sheet, df in sheets.items():
                print(f"\nSHEET: {sheet} | Rows: {len(df)}")
                print("Columns:", list(df.columns))
                print(df.head(3).to_string(index=False))
        else:
            df = pd.read_csv(path, low_memory=False)
            print("Rows:", len(df))
            print("Columns:", list(df.columns))
            print(df.head(3).to_string(index=False))
    except Exception as e:
        print("INSPECTION ERROR:", repr(e))


**[orig #88]** Read each sheet/table (tries encodings) and show the first rows  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/gw_framework")

FILES = {
    "Alwar district groundwater workbook": (
        ROOT / "data/Rajasthan_gw_data/Alwar.xlsx"
    ),
    "Alwar NHP workbook": (
        ROOT / "data/BDR/Alwar/nhpbdr 2023/Alwar NHP.xlsx"
    ),
    "Groundwater measurements with estimated heads": (
        ROOT / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/"
        "groundwater_measurements_with_estimated_head.csv"
    ),
    "Cleaned long seasonal groundwater observations": (
        ROOT / "data/Rajasthan_gw_data/"
        "Groundwater_Cleaning_According_to_User_Rules/"
        "groundwater_cleaned_long_seasonal.csv"
    ),
}

OUTPUT_DIR = ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inventory = []
errors = []

def inspect_dataframe(df, source_name, sheet_name="CSV"):
    print(f"\n  Sheet/table: {sheet_name}")
    print("  Shape:", df.shape)
    print("  Columns:", list(df.columns))
    print("\n  First 5 rows:")
    display(df.head())

    for col in df.columns:
        name = str(col).lower()
        if any(k in name for k in [
            "lat", "lon", "long", "coord", "well", "station",
            "date", "year", "month", "level", "depth", "head",
            "district", "block", "village", "elevation", "bgl"
        ]):
            print(f"  Candidate column: {col}")
            print("    Non-null:", int(df[col].notna().sum()))
            print("    Unique values:", int(df[col].nunique(dropna=True)))
            if pd.api.types.is_numeric_dtype(df[col]):
                print("    Min:", df[col].min())
                print("    Max:", df[col].max())

    inventory.append({
        "source": source_name,
        "sheet": sheet_name,
        "rows": len(df),
        "columns_count": len(df.columns),
        "columns": " | ".join(map(str, df.columns)),
    })

for source_name, path in FILES.items():
    print("\n" + "=" * 90)
    print(source_name)
    print("=" * 90)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        errors.append({"source": source_name, "error": "File not found"})
        continue

    try:
        print("File size (MB):", round(path.stat().st_size / (1024**2), 3))

        if path.suffix.lower() == ".csv":
            # Try common encodings without modifying the source.
            df = None
            for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
                try:
                    df = pd.read_csv(path, encoding=encoding, low_memory=False)
                    print("Read encoding:", encoding)
                    break
                except Exception:
                    pass

            if df is None:
                raise ValueError("Could not read CSV with attempted encodings.")

            inspect_dataframe(df, source_name)

        elif path.suffix.lower() in [".xlsx", ".xls"]:
            excel = pd.ExcelFile(path)
            print("Sheet names:", excel.sheet_names)

            for sheet in excel.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet)
                    inspect_dataframe(df, source_name, sheet)
                except Exception as e:
                    errors.append({
                        "source": source_name,
                        "sheet": sheet,
                        "error": str(e),
                    })
                    print("  SHEET ERROR:", sheet, str(e))

    except Exception as e:
        errors.append({"source": source_name, "error": str(e)})
        print("FILE ERROR:", repr(e))

if inventory:
    inventory_path = OUTPUT_DIR / "01_verified_source_inventory.csv"
    pd.DataFrame(inventory).to_csv(inventory_path, index=False)
    print("\nInventory saved to:", inventory_path)

if errors:
    error_path = OUTPUT_DIR / "02_inspection_errors.csv"
    pd.DataFrame(errors).to_csv(error_path, index=False)
    print("Inspection errors saved to:", error_path)

print("\nInspection complete. Original source files were not modified.")


**[orig #92]** Step 2 — normalise merge keys and keep original row numbers  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/gw_framework")

SEASONAL_FILE = (
    ROOT / "data/Rajasthan_gw_data/"
    "Groundwater_Cleaning_According_to_User_Rules/"
    "groundwater_cleaned_long_seasonal.csv"
)
HEAD_FILE = (
    ROOT / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/"
    "groundwater_measurements_with_estimated_head.csv"
)
OUT = (
    ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery/Step_2_QC"
)
OUT.mkdir(parents=True, exist_ok=True)

KEYS = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column"
]

seasonal = pd.read_csv(SEASONAL_FILE, encoding="utf-8-sig", low_memory=False)
heads = pd.read_csv(HEAD_FILE, encoding="utf-8-sig", low_memory=False)

# Check required columns.
for label, df in [("seasonal", seasonal), ("heads", heads)]:
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing merge keys: {missing}")

# Normalize merge keys consistently, without changing source files.
for df in [seasonal, heads]:
    for col in KEYS:
        df[col] = df[col].astype("string").str.strip()

# Keep a stable original row number for traceability.
seasonal["_seasonal_row"] = np.arange(len(seasonal))
heads["_head_row"] = np.arange(len(heads))

# Count repeated combinations of merge keys in each dataset.
seasonal["key_occurrence"] = (
    seasonal.groupby(KEYS, dropna=False, sort=False).cumcount() + 1
)
heads["key_occurrence"] = (
    heads.groupby(KEYS, dropna=False, sort=False).cumcount() + 1
)

seasonal_dup_mask = seasonal.duplicated(KEYS, keep=False)
heads_dup_mask = heads.duplicated(KEYS, keep=False)

seasonal_dups = seasonal.loc[seasonal_dup_mask].sort_values(KEYS)
heads_dups = heads.loc[heads_dup_mask].sort_values(KEYS)

seasonal_dups.to_csv(OUT / "08_duplicate_keys_seasonal.csv", index=False)
heads_dups.to_csv(OUT / "09_duplicate_keys_estimated_heads.csv", index=False)

print("=" * 72)
print("DUPLICATE KEY DIAGNOSTICS")
print("=" * 72)
print("Seasonal rows:", len(seasonal))
print("Estimated-head rows:", len(heads))
print("Seasonal rows participating in duplicate keys:", len(seasonal_dups))
print("Estimated-head rows participating in duplicate keys:", len(heads_dups))
print("Seasonal distinct six-key groups:", seasonal[KEYS].drop_duplicates().shape[0])
print("Head distinct six-key groups:", heads[KEYS].drop_duplicates().shape[0])

# Summarize duplicate groups, including the number of rows per key.
def duplicate_group_summary(df, name):
    counts = (
        df.groupby(KEYS, dropna=False)
        .size()
        .reset_index(name="rows_per_key")
    )
    counts = counts[counts["rows_per_key"] > 1].copy()
    counts.insert(0, "dataset", name)
    return counts

dup_summary = pd.concat([
    duplicate_group_summary(seasonal, "seasonal"),
    duplicate_group_summary(heads, "estimated_heads")
], ignore_index=True)

dup_summary.to_csv(OUT / "10_duplicate_key_group_summary.csv", index=False)

# Occurrence-indexed merge avoids a many-to-many Cartesian multiplication.
# This pairs the first occurrence with the first, second with second, etc.
# It is a diagnostic pairing, not proof that repeated rows are equivalent.
MERGE_KEYS = KEYS + ["key_occurrence"]

left_cols = KEYS + [
    "key_occurrence", "_seasonal_row", "WL_bgl", "WL_status"
]
right_cols = KEYS + [
    "key_occurrence", "_head_row",
    "Depth_to_water_bgl_m", "Estimated_head_m_AMSL",
    "DEM_ground_elevation_m", "Head_estimate_quality",
    "Elevation_status", "Elevation_review_flag"
]

left = seasonal[left_cols].copy()
right = heads[right_cols].copy()

left["WL_bgl_num"] = pd.to_numeric(left["WL_bgl"], errors="coerce")
right["DTW_num"] = pd.to_numeric(
    right["Depth_to_water_bgl_m"], errors="coerce"
)
right["Head_num"] = pd.to_numeric(
    right["Estimated_head_m_AMSL"], errors="coerce"
)
right["DEM_num"] = pd.to_numeric(
    right["DEM_ground_elevation_m"], errors="coerce"
)

merged = left.merge(
    right,
    on=MERGE_KEYS,
    how="outer",
    indicator=True,
    validate="one_to_one",
    suffixes=("_seasonal", "_head")
)

merged["dtw_difference_m"] = merged["WL_bgl_num"] - merged["DTW_num"]
merged["head_equation_difference_m"] = (
    merged["Head_num"] - (merged["DEM_num"] - merged["DTW_num"])
)

merged["dtw_mismatch_over_0_01m"] = (
    merged["dtw_difference_m"].abs() > 0.01
)
merged["head_equation_mismatch_over_0_01m"] = (
    merged["head_equation_difference_m"].abs() > 0.01
)

merged.to_csv(
    OUT / "11_occurrence_indexed_reconciliation.csv",
    index=False
)

print("\n" + "=" * 72)
print("RECONCILIATION RESULTS")
print("=" * 72)
print(merged["_merge"].value_counts(dropna=False).to_string())
print(
    "DTW mismatches > 0.01 m:",
    int(merged["dtw_mismatch_over_0_01m"].fillna(False).sum())
)
print(
    "Head equation mismatches > 0.01 m:",
    int(merged["head_equation_mismatch_over_0_01m"].fillna(False).sum())
)

# Save a compact summary of unmatched and discrepant records.
review_mask = (
    merged["_merge"].ne("both")
    | merged["dtw_mismatch_over_0_01m"].fillna(False)
    | merged["head_equation_mismatch_over_0_01m"].fillna(False)
)
merged.loc[review_mask].to_csv(
    OUT / "12_reconciliation_review_records.csv",
    index=False
)

print("\nOutputs saved under:", OUT)
print("Original source files were not modified.")


**[orig #93]** Step 2 — repeated keys: number occurrences, save with provenance  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/gw_framework")

HEAD_FILE = (
    ROOT / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/"
    "groundwater_measurements_with_estimated_head.csv"
)

OUT = (
    ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery/Step_2_QC"
)
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(HEAD_FILE, encoding="utf-8-sig", low_memory=False)

keys = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column"
]

for c in keys:
    df[c] = df[c].astype("string").str.strip()

value_cols = [
    c for c in [
        "Well_key", "Well_ID_source", "District", "Block",
        "WL_raw", "WL_status", "Depth_to_water_bgl_m",
        "Latitude", "Longitude", "DEM_ground_elevation_m",
        "Elevation_source", "Elevation_status",
        "Elevation_review_flag", "Estimated_head_m_AMSL",
        "Head_estimate_quality"
    ] if c in df.columns
]

# Preserve original order and number each repeated key occurrence.
df["_original_head_row"] = np.arange(len(df))
df["_occurrence"] = (
    df.groupby(keys, dropna=False, sort=False).cumcount() + 1
)

duplicates = df[df.duplicated(keys, keep=False)].copy()
extras = df[df["_occurrence"] > 1].copy()

print("All rows:", len(df))
print("Rows with repeated keys:", len(duplicates))
print("Additional occurrences beyond first:", len(extras))
print("Repeated key groups:", duplicates.groupby(keys, dropna=False).ngroups)

# Save the repeated rows with provenance.
duplicates.sort_values(keys + ["_occurrence"]).to_csv(
    OUT / "13_all_repeated_head_key_records.csv", index=False
)
extras.sort_values(keys + ["_occurrence"]).to_csv(
    OUT / "14_extra_head_occurrences.csv", index=False
)

# Compare values across occurrences within each key group.
records = []

for key_values, group in duplicates.groupby(keys, dropna=False, sort=False):
    record = dict(zip(keys, key_values if isinstance(key_values, tuple) else (key_values,)))
    record["occurrence_count"] = len(group)

    for col in value_cols:
        vals = group[col].astype("string").fillna("<NA>").unique().tolist()
        record[f"{col}_unique_count"] = len(vals)
        if len(vals) <= 5:
            record[f"{col}_values"] = " | ".join(vals)
        else:
            record[f"{col}_values"] = " | ".join(vals[:5]) + " | ..."

    records.append(record)

comparison = pd.DataFrame(records)
comparison.to_csv(
    OUT / "15_repeated_key_value_comparison.csv", index=False
)

print("\nValue variation across repeated-key groups:")
for col in value_cols:
    count_col = f"{col}_unique_count"
    if count_col in comparison:
        varied = int((comparison[count_col] > 1).sum())
        print(f"{col}: {varied} groups have multiple values")

print("\nSaved review outputs to:", OUT)
print("No rows were deleted.")


**[orig #94]** Step 2 — compare duplicate groups across all columns  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/gw_framework")

SOURCE = (
    ROOT / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/"
    "groundwater_measurements_with_estimated_head.csv"
)

OUT = (
    ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery/Step_2_QC"
)
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SOURCE, encoding="utf-8-sig", low_memory=False)

keys = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column"
]

# Compare duplicate groups using every original column.
dup_mask = df.duplicated(keys, keep=False)
duplicates = df.loc[dup_mask].copy()

# Count distinct full rows in each duplicate-key group.
full_row_counts = (
    duplicates.groupby(keys, dropna=False)
    .apply(lambda g: len(g.drop_duplicates()), include_groups=False)
)

conflicting_groups = int((full_row_counts > 1).sum())
repeated_groups = int(len(full_row_counts))
extra_occurrences = int(len(duplicates) - repeated_groups)

print("Total source rows:", len(df))
print("Repeated-key groups:", repeated_groups)
print("Extra occurrences:", extra_occurrences)
print("Groups with any full-row differences:", conflicting_groups)

if conflicting_groups > 0:
    print(
        "\nSTOP: Some repeated-key groups differ in at least one column. "
        "No deduplicated analysis copy was created."
    )
else:
    # All rows within each key group are identical.
    # Keep the first occurrence in a new file only.
    clean = df.drop_duplicates(subset=keys, keep="first").copy()

    target = OUT / "estimated_heads_deduplicated_analysis_copy.csv"
    clean.to_csv(target, index=False)

    report = {
        "source_file": str(SOURCE),
        "analysis_copy": str(target),
        "source_rows": int(len(df)),
        "analysis_copy_rows": int(len(clean)),
        "rows_removed_from_analysis_copy": int(len(df) - len(clean)),
        "repeated_key_groups": repeated_groups,
        "conflicting_duplicate_groups": conflicting_groups,
        "deduplication_basis": (
            "Six observation provenance keys; all columns verified identical "
            "within repeated-key groups."
        ),
        "original_file_modified": False
    }

    with open(OUT / "16_duplicate_resolution_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print("\nAnalysis copy created:", target)
    print(json.dumps(report, indent=2))

print("\nOriginal source remains unchanged:", SOURCE)


**[orig #95]** Step 3 — paths and observation-level diagnostics  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ---------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------
INPUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_3_Spatial_Head_Quality"
)
OUT.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise FileNotFoundError(
        f"Input not found: {INPUT}\n"
        "Check the path in Google Drive before proceeding."
    )

df = pd.read_csv(INPUT, low_memory=False)
df.columns = df.columns.str.strip()

print("Input shape:", df.shape)
print("Columns:", df.columns.tolist())

# ---------------------------------------------------------
# 2. REQUIRED COLUMNS
# ---------------------------------------------------------
required = [
    "Latitude", "Longitude", "Depth_to_water_bgl_m",
    "DEM_ground_elevation_m", "Estimated_head_m_AMSL",
    "Elevation_review_flag", "Coordinate_valid_Rajasthan_bounds"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

# Convert relevant fields to numeric without changing source file.
numeric_cols = [
    "Latitude", "Longitude", "Depth_to_water_bgl_m",
    "DEM_ground_elevation_m", "Estimated_head_m_AMSL"
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---------------------------------------------------------
# 3. SPATIAL SCREENING
# ---------------------------------------------------------
# Broad screening envelope only; NOT an official district boundary.
# Replace this test with an authoritative Alwar polygon test
# when the official boundary shapefile/GeoJSON is available.
df["inside_provisional_alwar_envelope"] = (
    df["Latitude"].between(27.0, 28.2)
    & df["Longitude"].between(76.0, 77.3)
)

df["coordinate_missing"] = (
    df["Latitude"].isna() | df["Longitude"].isna()
)
df["coordinate_outside_india_reasonable_bounds"] = (
    ~df["Latitude"].between(6, 38)
    | ~df["Longitude"].between(67, 98)
)
df["spatial_review_flag"] = np.select(
    [
        df["coordinate_missing"],
        df["coordinate_outside_india_reasonable_bounds"],
        ~df["inside_provisional_alwar_envelope"]
    ],
    [
        "REVIEW_MISSING_COORDINATE",
        "REVIEW_IMPLAUSIBLE_INDIA_COORDINATE",
        "REVIEW_OUTSIDE_PROVISIONAL_ALWAR_ENVELOPE"
    ],
    default="PROVISIONAL_ENVELOPE_PASS"
)

# ---------------------------------------------------------
# 4. DTW AND WELL-DEPTH QUALITY FLAGS
# ---------------------------------------------------------
dtw = df["Depth_to_water_bgl_m"]

df["dtw_missing"] = dtw.isna()
df["dtw_negative_flag"] = dtw < 0
df["dtw_zero_flag"] = dtw == 0
df["dtw_over_100m_flag"] = dtw > 100

# Use the source well depth if present.
depth_col = next(
    (c for c in ["Total_Depth_bgl", "well_depth_m", "Well_Depth_m"]
     if c in df.columns),
    None
)

if depth_col:
    df[depth_col] = pd.to_numeric(df[depth_col], errors="coerce")
    df["well_depth_missing"] = df[depth_col].isna()
    df["dtw_exceeds_well_depth_flag"] = (
        dtw.notna()
        & df[depth_col].notna()
        & (dtw > df[depth_col])
    )
else:
    df["well_depth_missing"] = True
    df["dtw_exceeds_well_depth_flag"] = pd.NA
    print(
        "WARNING: No recognized well-depth column found. "
        "Well-depth consistency cannot be evaluated."
    )

# ---------------------------------------------------------
# 5. RECOMPUTE GROUNDWATER HEAD
# Head elevation = ground elevation - depth to water below ground.
# ---------------------------------------------------------
df["head_recomputed_m_AMSL"] = (
    df["DEM_ground_elevation_m"] - dtw
)
df["head_difference_m"] = (
    df["Estimated_head_m_AMSL"] - df["head_recomputed_m_AMSL"]
)
df["head_equation_mismatch_flag"] = (
    df["head_difference_m"].abs() > 0.01
)

df["ground_elevation_missing"] = (
    df["DEM_ground_elevation_m"].isna()
)
df["estimated_head_missing"] = (
    df["Estimated_head_m_AMSL"].isna()
)

# Do not label DEM-based elevations as surveyed.
df["elevation_source_review_flag"] = (
    df["Elevation_review_flag"].astype("string")
    .fillna("MISSING_ELEVATION_REVIEW_FLAG")
)

# ---------------------------------------------------------
# 6. OBSERVATION KEY DUPLICATES
# ---------------------------------------------------------
key_candidates = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column"
]
key_cols = [c for c in key_candidates if c in df.columns]

if key_cols:
    df["duplicate_observation_key_flag"] = df.duplicated(
        subset=key_cols, keep=False
    )
else:
    df["duplicate_observation_key_flag"] = False
    print("WARNING: Full source observation key unavailable.")

# ---------------------------------------------------------
# 7. COMBINED REVIEW CLASS
# Flags are retained; no observation is automatically deleted.
# ---------------------------------------------------------
flag_cols = [
    "coordinate_missing",
    "coordinate_outside_india_reasonable_bounds",
    "dtw_missing",
    "dtw_negative_flag",
    "dtw_zero_flag",
    "dtw_over_100m_flag",
    "dtw_exceeds_well_depth_flag",
    "head_equation_mismatch_flag",
    "ground_elevation_missing",
    "estimated_head_missing",
    "duplicate_observation_key_flag"
]

def has_flag(row):
    for col in flag_cols:
        val = row.get(col, False)
        if pd.notna(val) and bool(val):
            return True
    return False

df["needs_manual_review"] = df.apply(has_flag, axis=1)

# ---------------------------------------------------------
# 8. EXPORT AUDIT FILES
# ---------------------------------------------------------
df.to_csv(OUT / "01_spatial_head_quality_all_records.csv", index=False)

df.loc[df["needs_manual_review"]].to_csv(
    OUT / "02_records_requiring_review.csv", index=False
)

df.loc[~df["inside_provisional_alwar_envelope"]].to_csv(
    OUT / "03_outside_provisional_envelope.csv", index=False
)

df.loc[df["head_equation_mismatch_flag"]].to_csv(
    OUT / "04_head_equation_mismatches.csv", index=False
)

# Numeric summary
summary = {
    "input_rows": int(len(df)),
    "unique_wells": int(
        df["Well_key"].nunique()
        if "Well_key" in df.columns else 0
    ),
    "coordinate_missing_rows": int(df["coordinate_missing"].sum()),
    "inside_provisional_envelope_rows": int(
        df["inside_provisional_alwar_envelope"].sum()
    ),
    "outside_or_missing_envelope_rows": int(
        (~df["inside_provisional_alwar_envelope"]).sum()
    ),
    "negative_dtw_rows": int(df["dtw_negative_flag"].sum()),
    "zero_dtw_rows": int(df["dtw_zero_flag"].sum()),
    "dtw_over_100m_rows": int(df["dtw_over_100m_flag"].sum()),
    "dtw_exceeds_well_depth_rows": int(
        df["dtw_exceeds_well_depth_flag"].fillna(False).sum()
    ),
    "head_equation_mismatch_rows": int(
        df["head_equation_mismatch_flag"].fillna(False).sum()
    ),
    "missing_ground_elevation_rows": int(
        df["ground_elevation_missing"].sum()
    ),
    "missing_estimated_head_rows": int(
        df["estimated_head_missing"].sum()
    ),
    "duplicate_observation_key_rows": int(
        df["duplicate_observation_key_flag"].sum()
    ),
    "records_requiring_review": int(df["needs_manual_review"].sum()),
    "spatial_test_warning": (
        "Provisional bounding box only; not official district boundary."
    )
}

with open(OUT / "05_audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Flag counts
rows = []
for col in flag_cols + ["spatial_review_flag"]:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        for value, count in counts.items():
            rows.append({
                "flag": col,
                "value": str(value),
                "count": int(count)
            })

pd.DataFrame(rows).to_csv(
    OUT / "06_flag_counts.csv", index=False
)

print("\nAUDIT SUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nOutputs saved to:", OUT)


**[orig #96]** Step 3A/3B — missing-head reasons; district summary  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_3_Spatial_Head_Quality"
)

INPUT = BASE / "01_spatial_head_quality_all_records.csv"
OUT = BASE / "Step_3A_Diagnostic_Reports"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)

# Numeric conversions
for col in [
    "Latitude", "Longitude", "Depth_to_water_bgl_m",
    "DEM_ground_elevation_m", "Estimated_head_m_AMSL"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 1. Missing-head reason for every observation
df["head_missing_reason"] = np.select(
    [
        df["Depth_to_water_bgl_m"].isna()
        & df["DEM_ground_elevation_m"].isna(),

        df["Depth_to_water_bgl_m"].isna(),

        df["DEM_ground_elevation_m"].isna(),

        df["Estimated_head_m_AMSL"].isna()
    ],
    [
        "DTW_AND_ELEVATION_MISSING",
        "DTW_MISSING",
        "GROUND_ELEVATION_MISSING",
        "HEAD_MISSING_OTHER_REASON"
    ],
    default="HEAD_AVAILABLE"
)

df["head_missing_reason"].value_counts(dropna=False).rename_axis(
    "reason"
).reset_index(name="records").to_csv(
    OUT / "01_head_missing_reason_counts.csv", index=False
)

# 2. Geographic summary by district
geo = df.groupby("District", dropna=False).agg(
    records=("Well_key", "size"),
    unique_wells=("Well_key", "nunique"),
    missing_coordinates=("coordinate_missing", "sum"),
    inside_provisional_envelope=(
        "inside_provisional_alwar_envelope", "sum"
    ),
    numeric_dtw=("Depth_to_water_bgl_m", "count"),
    available_heads=("Estimated_head_m_AMSL", "count")
).reset_index()

geo["provisional_inside_record_percent"] = (
    100 * geo["inside_provisional_envelope"] / geo["records"]
)
geo.to_csv(OUT / "02_district_spatial_summary.csv", index=False)

# 3. One row per well: check whether location stays consistent
well_summary = df.groupby("Well_key", dropna=False).agg(
    Well_ID_source=("Well_ID_source", "first"),
    District=("District", "first"),
    Block=("Block", "first"),
    latitude=("Latitude", "median"),
    longitude=("Longitude", "median"),
    latitude_unique=("Latitude", "nunique"),
    longitude_unique=("Longitude", "nunique"),
    records=("Well_key", "size"),
    numeric_dtw=("Depth_to_water_bgl_m", "count"),
    available_heads=("Estimated_head_m_AMSL", "count"),
    negative_dtw_records=("dtw_negative_flag", "sum"),
    zero_dtw_records=("dtw_zero_flag", "sum"),
    deep_dtw_records=("dtw_over_100m_flag", "sum"),
    missing_coordinate_records=("coordinate_missing", "sum"),
    inside_envelope_records=(
        "inside_provisional_alwar_envelope", "sum"
    )
).reset_index()

well_summary["coordinate_changes_flag"] = (
    (well_summary["latitude_unique"] > 1)
    | (well_summary["longitude_unique"] > 1)
)
well_summary["no_numeric_dtw_flag"] = (
    well_summary["numeric_dtw"] == 0
)
well_summary["no_estimated_head_flag"] = (
    well_summary["available_heads"] == 0
)
well_summary.to_csv(OUT / "03_well_level_review.csv", index=False)

# 4. Coordinate ranges and district labels
coord_summary = df.groupby("District", dropna=False).agg(
    min_lat=("Latitude", "min"),
    max_lat=("Latitude", "max"),
    min_lon=("Longitude", "min"),
    max_lon=("Longitude", "max")
).reset_index()

coord_summary.to_csv(
    OUT / "04_district_coordinate_ranges.csv", index=False
)

# 5. Review records flagged by the audit
review_cols = [
    c for c in [
        "Well_key", "Well_ID_source", "District", "Block",
        "Source_file", "Source_sheet", "Source_row_number",
        "Year", "Season", "Latitude", "Longitude",
        "Depth_to_water_bgl_m", "DEM_ground_elevation_m",
        "Estimated_head_m_AMSL", "spatial_review_flag",
        "Elevation_review_flag", "Head_estimate_quality"
    ] if c in df.columns
]
df.loc[
    df["needs_manual_review"].fillna(False), review_cols
].to_csv(OUT / "05_manual_review_queue.csv", index=False)

print("DIAGNOSTICS COMPLETE")
print("Records:", len(df))
print("Unique wells:", df["Well_key"].nunique())
print("\nHead availability:")
print(df["head_missing_reason"].value_counts(dropna=False))
print("\nDistrict summary:")
print(geo.to_string(index=False))
print("\nWell-level output:", OUT / "03_well_level_review.csv")
print("All reports saved to:", OUT)


**[orig #97]** Step 3B — Alwar status categories vs numeric DTW  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_3_Spatial_Head_Quality"
)
INPUT = BASE / "01_spatial_head_quality_all_records.csv"
OUT = BASE / "Step_3B_Alwar_DTW_Review"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)

# Select Alwar using the recorded district label.
alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

alwar["Depth_to_water_bgl_m"] = pd.to_numeric(
    alwar["Depth_to_water_bgl_m"], errors="coerce"
)
alwar["Year"] = pd.to_numeric(alwar["Year"], errors="coerce")

print("Alwar records:", len(alwar))
print("Alwar wells:", alwar["Well_key"].nunique())

# 1. Raw status categories
status = (
    alwar.groupby(["WL_status"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)
status.to_csv(OUT / "01_alwar_status_counts.csv", index=False)

# 2. Raw status versus numeric DTW
alwar["dtw_available"] = alwar["Depth_to_water_bgl_m"].notna()
status_dtw = (
    alwar.groupby(["WL_status"], dropna=False)
    .agg(
        records=("Well_key", "size"),
        numeric_dtw=("dtw_available", "sum"),
        wells=("Well_key", "nunique")
    )
    .reset_index()
)
status_dtw["missing_dtw"] = (
    status_dtw["records"] - status_dtw["numeric_dtw"]
)
status_dtw.to_csv(
    OUT / "02_alwar_status_vs_dtw.csv", index=False
)

# 3. Inspect actual raw values for missing DTW
missing = alwar[~alwar["dtw_available"]].copy()

raw_values = (
    missing.groupby(["WL_status", "WL_raw"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)
raw_values.to_csv(
    OUT / "03_missing_dtw_raw_value_patterns.csv", index=False
)

# 4. Coverage by year and season
coverage = (
    alwar.groupby(["Year", "Season"], dropna=False)
    .agg(
        records=("Well_key", "size"),
        wells=("Well_key", "nunique"),
        numeric_dtw=("dtw_available", "sum")
    )
    .reset_index()
)
coverage["dtw_coverage_percent"] = (
    100 * coverage["numeric_dtw"] / coverage["records"]
)
coverage.to_csv(
    OUT / "04_alwar_year_season_coverage.csv", index=False
)

# 5. Per-well observation coverage
well = (
    alwar.groupby("Well_key", dropna=False)
    .agg(
        Well_ID_source=("Well_ID_source", "first"),
        Block=("Block", "first"),
        records=("Well_key", "size"),
        numeric_dtw=("dtw_available", "sum"),
        first_year=("Year", "min"),
        last_year=("Year", "max")
    )
    .reset_index()
)
well["dtw_coverage_percent"] = (
    100 * well["numeric_dtw"] / well["records"]
)
well.to_csv(OUT / "05_alwar_well_coverage.csv", index=False)

# 6. Export a sample of missing observations for manual inspection
sample_cols = [
    c for c in [
        "Well_key", "Well_ID_source", "Block", "Village",
        "Source_file", "Source_sheet", "Source_row_number",
        "Year", "Season", "Original_WL_Column",
        "WL_raw", "WL_status", "Depth_to_water_bgl_m",
        "Latitude", "Longitude", "DEM_ground_elevation_m",
        "Estimated_head_m_AMSL"
    ] if c in missing.columns
]
missing[sample_cols].head(500).to_csv(
    OUT / "06_first_500_missing_dtw_records.csv", index=False
)

print("\nSTATUS COUNTS")
print(status.to_string(index=False))

print("\nSTATUS VS DTW")
print(status_dtw.to_string(index=False))

print("\nAlwar DTW coverage:",
      round(100 * alwar["dtw_available"].mean(), 2), "%")
print("\nSaved reports to:", OUT)


**[orig #99]** Step 4 — Alwar well identifiers and status labels  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------
INPUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_4_Observation_Coverage"
)
OUT.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT}")

df = pd.read_csv(INPUT, low_memory=False)
df.columns = df.columns.str.strip()

print("Input shape:", df.shape)
print("Columns:", df.columns.tolist())

# ---------------------------------------------------------
# 2. VALIDATE REQUIRED COLUMNS
# ---------------------------------------------------------
required = [
    "District", "Well_key", "Well_ID_source",
    "Year", "Season", "WL_status",
    "Depth_to_water_bgl_m"
]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# ---------------------------------------------------------
# 3. SELECT ALWAR
# ---------------------------------------------------------
alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

alwar["Depth_to_water_bgl_m"] = pd.to_numeric(
    alwar["Depth_to_water_bgl_m"], errors="coerce"
)
alwar["Year"] = pd.to_numeric(alwar["Year"], errors="coerce")

alwar["status_clean"] = (
    alwar["WL_status"].astype("string").str.strip().str.casefold()
)
alwar["dtw_available"] = alwar["Depth_to_water_bgl_m"].notna()

print("\nAlwar records:", len(alwar))
print("Alwar unique wells:", alwar["Well_key"].nunique())

# ---------------------------------------------------------
# 4. WELL-LEVEL OBSERVATION COVERAGE
# ---------------------------------------------------------
well = (
    alwar.groupby("Well_key", dropna=False)
    .agg(
        Well_ID_source=("Well_ID_source", "first"),
        District=("District", "first"),
        Block=("Block", "first"),
        records=("Well_key", "size"),
        numeric_readings=("dtw_available", "sum"),
        first_year=("Year", "min"),
        last_year=("Year", "max"),
        seasons_observed=("Season", "nunique")
    )
    .reset_index()
)

status_counts = pd.crosstab(
    alwar["Well_key"], alwar["status_clean"]
).reset_index()

well = well.merge(status_counts, on="Well_key", how="left")

for col in ["numeric reading", "dry", "no reading"]:
    if col not in well.columns:
        well[col] = 0

well["numeric_coverage_percent"] = (
    100 * well["numeric_readings"] / well["records"]
)

well["well_class"] = np.select(
    [
        (well["numeric_readings"] == 0) & (well["dry"] > 0),
        (well["numeric_readings"] == 0) & (well["no reading"] > 0),
        (well["numeric_readings"] > 0) & (well["dry"] > 0),
        (well["numeric_readings"] > 0)
    ],
    [
        "NO_NUMERIC_DTW_WITH_DRY_STATUS",
        "NO_NUMERIC_DTW_WITH_MISSING_STATUS",
        "NUMERIC_DTW_AND_DRY_STATUS",
        "NUMERIC_DTW_PRESENT"
    ],
    default="REVIEW_STATUS"
)

well.to_csv(
    OUT / "01_alwar_well_observation_summary.csv", index=False
)

# ---------------------------------------------------------
# 5. YEAR-SEASON COVERAGE
# ---------------------------------------------------------
coverage = (
    alwar.groupby(["Year", "Season"], dropna=False)
    .agg(
        records=("Well_key", "size"),
        wells=("Well_key", "nunique"),
        numeric_readings=("dtw_available", "sum")
    )
    .reset_index()
)

coverage["numeric_coverage_percent"] = (
    100 * coverage["numeric_readings"] / coverage["records"]
)

coverage.to_csv(
    OUT / "02_alwar_year_season_coverage.csv", index=False
)

# ---------------------------------------------------------
# 6. STATUS COUNTS
# ---------------------------------------------------------
status_counts_all = (
    alwar.groupby("status_clean", dropna=False)
    .agg(
        records=("Well_key", "size"),
        unique_wells=("Well_key", "nunique"),
        numeric_dtw=("dtw_available", "sum")
    )
    .reset_index()
)

status_counts_all.to_csv(
    OUT / "03_alwar_status_counts.csv", index=False
)

# ---------------------------------------------------------
# 7. ANNUAL STATUS COUNTS
# ---------------------------------------------------------
annual = (
    alwar.groupby(["Year", "status_clean"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

annual.to_csv(
    OUT / "04_alwar_annual_status_counts.csv", index=False
)

# ---------------------------------------------------------
# 8. SUMMARY
# ---------------------------------------------------------
summary = {
    "alwar_records": int(len(alwar)),
    "unique_wells": int(alwar["Well_key"].nunique()),
    "numeric_dtw_records": int(alwar["dtw_available"].sum()),
    "wells_with_numeric_dtw": int(
        (well["numeric_readings"] > 0).sum()
    ),
    "wells_without_numeric_dtw": int(
        (well["numeric_readings"] == 0).sum()
    ),
    "wells_with_numeric_and_dry_status": int(
        (
            (well["numeric_readings"] > 0)
            & (well["dry"] > 0)
        ).sum()
    ),
    "wells_with_only_one_numeric_reading": int(
        (well["numeric_readings"] == 1).sum()
    ),
    "wells_with_at_least_5_numeric_readings": int(
        (well["numeric_readings"] >= 5).sum()
    ),
    "overall_numeric_coverage_percent": round(
        100 * alwar["dtw_available"].mean(), 2
    )
}

pd.Series(summary, name="value").to_csv(
    OUT / "05_alwar_coverage_summary.csv"
)

print("\nSUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSTATUS COUNTS")
print(status_counts_all.to_string(index=False))

print("\nOutputs saved to:", OUT)


**[orig #100]** Step 5 — year/season coverage of Alwar records  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

INPUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_5_Temporal_Coverage"
)
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)

# Select Alwar records
alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

alwar["Year"] = pd.to_numeric(alwar["Year"], errors="coerce")
alwar["Depth_to_water_bgl_m"] = pd.to_numeric(
    alwar["Depth_to_water_bgl_m"], errors="coerce"
)

# Map year and season to a comparable approximate date.
# This is for coverage diagnostics, not a replacement for
# the original field measurement date.
season_map = {
    "pre": 4,
    "pre-monsoon": 4,
    "pre monsoon": 4,
    "pre_monsoon": 4,
    "post": 11,
    "post-monsoon": 11,
    "post monsoon": 11,
    "post_monsoon": 11,
    "monsoon": 8,
    "winter": 1
}

season_norm = (
    alwar["Season"].astype("string").str.strip().str.casefold()
)
alwar["approx_month"] = season_norm.map(season_map)

# Fall back to the original WL column label if season names
# use a different format.
if "Original_WL_Column" in alwar.columns:
    original = (
        alwar["Original_WL_Column"]
        .astype("string").str.strip().str.casefold()
    )
    alwar.loc[
        alwar["approx_month"].isna()
        & original.str.contains("pre", na=False),
        "approx_month"
    ] = 4
    alwar.loc[
        alwar["approx_month"].isna()
        & original.str.contains("post", na=False),
        "approx_month"
    ] = 11

alwar["approx_date"] = pd.to_datetime(
    dict(
        year=alwar["Year"],
        month=alwar["approx_month"],
        day=15
    ),
    errors="coerce"
)

# Numeric observations only for temporal diagnostics
obs = alwar[
    alwar["Depth_to_water_bgl_m"].notna()
    & alwar["approx_date"].notna()
].copy()

obs = obs.sort_values(["Well_key", "approx_date"])

# Within-well gap in days between numeric observations
obs["gap_days"] = (
    obs.groupby("Well_key")["approx_date"]
    .diff().dt.days
)

# Per-well summary
well_rows = []

for well_id, g in alwar.groupby("Well_key", dropna=False):
    numeric = obs[obs["Well_key"] == well_id].sort_values("approx_date")
    dates = numeric["approx_date"].dropna()

    numeric_count = len(numeric)
    if numeric_count >= 2:
        gaps = dates.diff().dt.days.dropna()
        median_gap = float(gaps.median()) if len(gaps) else np.nan
        max_gap = float(gaps.max()) if len(gaps) else np.nan
        span_years = (dates.max() - dates.min()).days / 365.25
    else:
        median_gap = np.nan
        max_gap = np.nan
        span_years = 0.0 if numeric_count == 1 else np.nan

    statuses = (
        g["WL_status"].astype("string").str.strip().str.casefold()
    )

    well_rows.append({
        "Well_key": well_id,
        "Well_ID_source": g["Well_ID_source"].iloc[0],
        "Block": g["Block"].iloc[0],
        "all_records": len(g),
        "numeric_observations": numeric_count,
        "dry_records": int((statuses == "dry").sum()),
        "no_reading_records": int((statuses == "no reading").sum()),
        "first_numeric_date_approx": dates.min() if len(dates) else pd.NaT,
        "last_numeric_date_approx": dates.max() if len(dates) else pd.NaT,
        "numeric_span_years": span_years,
        "median_gap_days": median_gap,
        "maximum_gap_days": max_gap,
        "seasons_with_numeric_data": (
            numeric["Season"].nunique() if numeric_count else 0
        )
    })

well_summary = pd.DataFrame(well_rows)

well_summary["coverage_screen"] = np.select(
    [
        well_summary["numeric_observations"].eq(0),
        well_summary["numeric_observations"].lt(5),
        well_summary["numeric_span_years"].lt(3),
        well_summary["maximum_gap_days"].gt(5 * 365)
    ],
    [
        "NO_NUMERIC_OBSERVATIONS",
        "FEWER_THAN_5_NUMERIC_OBSERVATIONS",
        "SHORT_NUMERIC_DATE_SPAN",
        "LONG_GAP_OVER_5_YEARS"
    ],
    default="FURTHER_TEMPORAL_REVIEW"
)

well_summary.to_csv(
    OUT / "01_alwar_well_temporal_coverage.csv", index=False
)

obs.to_csv(
    OUT / "02_alwar_numeric_observations_with_gaps.csv", index=False
)

# Annual coverage summary
annual = (
    alwar.groupby("Year", dropna=False)
    .agg(
        records=("Well_key", "size"),
        wells=("Well_key", "nunique"),
        numeric_records=("Depth_to_water_bgl_m", "count")
    )
    .reset_index()
)
annual["numeric_coverage_percent"] = (
    100 * annual["numeric_records"] / annual["records"]
)
annual.to_csv(OUT / "03_alwar_annual_coverage.csv", index=False)

print("ALWAR TEMPORAL AUDIT")
print("Wells:", well_summary["Well_key"].nunique())
print("Wells with numeric observations:",
      int((well_summary["numeric_observations"] > 0).sum()))
print("Wells with >=5 numeric observations:",
      int((well_summary["numeric_observations"] >= 5).sum()))
print("Wells with >=10 years numeric span:",
      int((well_summary["numeric_span_years"] >= 10).sum()))
print("\nCoverage screen:")
print(well_summary["coverage_screen"].value_counts().to_string())
print("\nSaved to:", OUT)


**[orig #101]** Step 6 — duplicate numeric observations within well/year/season; well-level summary  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

INPUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_6_Trend_Candidate_QC"
)
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)

alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

alwar["Year"] = pd.to_numeric(alwar["Year"], errors="coerce")
alwar["Depth_to_water_bgl_m"] = pd.to_numeric(
    alwar["Depth_to_water_bgl_m"], errors="coerce"
)

numeric = alwar[
    alwar["Depth_to_water_bgl_m"].notna()
    & alwar["Year"].notna()
].copy()

# Detect duplicate numeric observations within well/year/season.
key = ["Well_key", "Year", "Season"]
numeric["duplicate_well_year_season"] = numeric.duplicated(
    subset=key, keep=False
)

duplicate_groups = (
    numeric.loc[numeric["duplicate_well_year_season"]]
    .sort_values(key)
)
duplicate_groups.to_csv(
    OUT / "01_duplicate_numeric_well_year_season.csv", index=False
)

# Well-level numeric observation summary.
summary = numeric.groupby("Well_key").agg(
    Well_ID_source=("Well_ID_source", "first"),
    Block=("Block", "first"),
    numeric_readings=("Depth_to_water_bgl_m", "count"),
    first_year=("Year", "min"),
    last_year=("Year", "max"),
    unique_numeric_years=("Year", "nunique"),
    unique_numeric_seasons=("Season", "nunique"),
    negative_dtw_count=("Depth_to_water_bgl_m", lambda s: (s < 0).sum()),
    zero_dtw_count=("Depth_to_water_bgl_m", lambda s: (s == 0).sum())
).reset_index()

summary["numeric_year_span"] = (
    summary["last_year"] - summary["first_year"]
)

duplicate_wells = set(
    duplicate_groups["Well_key"].dropna().unique()
)
summary["duplicate_year_season_flag"] = (
    summary["Well_key"].isin(duplicate_wells)
)

summary["initial_trend_screen_pass"] = (
    (summary["numeric_readings"] >= 10)
    & (summary["numeric_year_span"] >= 10)
    & (~summary["duplicate_year_season_flag"])
)

summary["trend_screen_status"] = np.where(
    summary["initial_trend_screen_pass"],
    "CANDIDATE_FOR_FURTHER_REVIEW",
    "NOT_YET_PASSED_INITIAL_SCREEN"
)

summary.to_csv(
    OUT / "02_well_trend_screen_summary.csv", index=False
)

candidates = summary[
    summary["initial_trend_screen_pass"]
].copy()

candidates.to_csv(
    OUT / "03_initial_trend_candidates.csv", index=False
)

print("Numeric Alwar observations:", len(numeric))
print("Wells with numeric observations:", summary["Well_key"].nunique())
print("Wells with duplicate numeric well/year/season keys:",
      summary["duplicate_year_season_flag"].sum())
print("Initial trend candidates:", len(candidates))
print("\nCandidate numeric reading count:")
print(candidates["numeric_readings"].describe())
print("\nCandidate year-span summary:")
print(candidates["numeric_year_span"].describe())
print("\nOutputs saved to:", OUT)
